# 암종 분류 v13 — "실제로 돌아가게 만들고, 새 피처는 스스로 검증해서 켠다"

## v13에서 가장 먼저 고친 것 (점수와 직결)
v10 → v11 → v12 노트북의 저장된 출력을 비교하면 **세 버전의 점수가 소수점까지 같습니다** (OOF macroF1 0.4714, 중복 규칙 후 0.5373).
이유는 두 가지입니다.
1. `lightgbm 미설치 -> USE_LGBM=False` 메시지가 출력에 찍혀 있습니다. v11의 LightGBM은 **한 번도 실행되지 않았습니다.**
2. v12의 생물학 피처 4종은 전부 기본 OFF였고, ablation(`RUN_ABLATION3`)도 꺼져 있어서 **한 번도 효과가 측정되지 않았습니다.**

즉 v10 이후 "개선"은 모두 코드로만 존재했고 점수에는 반영된 적이 없습니다. v13은 이를 실제로 돌리고, 효과가 확인된 것만 켭니다.

## v13 변경점
- **LightGBM 자동 설치** (첫 셀). 미설치로 조용히 빠지지 않게 함.
- **시드 평균**(`SEEDS`): XGB/LGBM을 여러 시드로 돌려 평균 (분산 감소).
- **행 정규화 LR(`lrn`)**: TMB가 0~2393으로 퍼져 있어서, 변이가 많은 샘플이 로짓을 지배함. L2 행 정규화 후 LR.
- **새 피처 3종** (전부 기본 OFF, 자동 ablation으로 선택): `USE_GENE_SPLIT`(유전자별 비침묵/침묵 분리), `USE_BLOSUM`(BLOSUM62), `USE_NB_STACK`(나이브베이즈 사후확률을 내부 그룹 OOF로 트리 모델에 공급).
- **자동 ablation**(`AUTO_ABLATION`): v12의 생물학 피처 4종 + 새 피처 3종을 그룹 분할 CV로 하나씩 켜 보고, +0.003 이상인 것만 채택. 합쳤을 때 오히려 나빠지면 가장 좋은 하나만 유지.
- **앙상블 가중치 검증을 중첩(nested) CV로**: 가중치를 다른 fold에서 고르고 남은 fold로 평가한 점수로 모델 조합을 선택. (md 리포트의 "Inner/Outer 분리" 항목 반영)
- **프로필 해시 정규화**(`NORM_HASH`): canon + 토큰 정렬 후 해시.
- **분포 점검 두 가지 추가**: ① `other` 토큰 실태 출력, ② train/test 적대적 검증(AUC).
- **라벨 분포 이동 보정(BBSE)** (`USE_LABEL_SHIFT`): 시뮬레이션에서 이득이 확인될 때만 `_shift` 제출본을 **별도 파일**로 추가 생성. 기존 `_core`/`_dup`은 그대로 유지.

## 알아 둘 점
- v12 md 리포트의 "`other` 변이 세분화"는 기대 이득이 거의 없습니다. 저장된 출력에서 `other`는 전체 25만 토큰 중 319개(0.1%), splice 0개, inframe 3개입니다. 진단 셀로 실태만 확인합니다.
- IntOGen/CIViC 같은 외부 DB 연계는 대회의 외부 데이터 규정 확인이 먼저라서 넣지 않았습니다.
- test 예측 분포가 train 분포와 크게 다릅니다 (v12 출력: COAD 548행, STES 377행 / train 비율로는 각각 약 92행, 156행). 적대적 검증 셀과 라벨 이동 셀이 이를 다룹니다.

---

(아래는 v12까지의 변경 이력)

# 암종 분류 v6 (v3 + 쌍둥이 규칙 + 표기 통일): 이진 행렬 + 파생변수 + 인프레임 핫스팟 + 동반변이 콤보 + XGB/LR 앙상블 (5-fold CV)

**v2 대비 변경점**
- 인프레임 변이(`EGFR E746_A750del` 등)를 핫스팟 토큰으로 추가 (`gene|inframe`, `gene|위치_inframe`)
- 가이드 기반 동반변이 콤보 피처 (`cb_*`), TMB 기준 5 / 30
- **EM 사전확률 보정 제거** (macroF1을 0.46 -> 0.21로 망가뜨림). COAD 확률 보정은 검토 후 이득이 기준 미만(미적용)이고 test 분포 정보가 필요해서 **삭제**
- 실행 순서 버그 수정: 예전 prior 셀이 CV보다 앞에 있어 `oof_ens`가 정의되기 전에 쓰였음 -> 앙상블 이후로 이동
- CV를 함수(`run_cv`)로 묶고, 콤보 ON/OFF 비교(ablation) 셀 추가

**패키지**: `pip install pandas numpy scikit-learn xgboost`


**v5 변경점**: 쌍둥이(형제 라벨) 규칙 추가 (5-2절). KIRC↔KIPAN, LGG↔GBMLGG 는 같은 환자가 두 코호트에 중복 등록된 경우가 있어, train에 짝 라벨이 있는 프로필이 test에 나오면 짝 암종으로 교정한다. (최소빈도는 v3 그대로 5)

**v6 주요 변경점**: 변이 표기 정규화 (`NORMALIZE_TOKENS`, `canon`)
- HGVS 규약상 같은 변이가 `R1450X` / `R1450*` / `p.Arg1450Ter` 등으로 달리 적힐 수 있어, 표기와 무관하게 같은 변이가 같은 피처가 되도록 정규화하고 한 칸 안의 중복 표기를 제거한다.
- 모든 연산은 토큰/행 하나만 보는 독립 연산이며 train/test에 동일하게 적용된다. test 통계는 사용하지 않는다.
- 단, train에는 구식 표기가 0건이라 train만으로 효과를 검증할 수 없다. 운영진 확인 전까지는 `NORMALIZE_TOKENS=False` 제출본(`nonorm`)을 함께 준비한다.
- **v7 보강(canon)**: 괄호 `p.(Arg1450Ter)`, 대문자 `P.`, 소문자 `x`/`ter`, 대소문자 섞인 3글자 표기도 같은 변이로 통일. train에는 해당 표기가 0건이라 기존 결과는 변하지 않는다.

**v8 변경점 (TwinRuleRefine.md 반영)**: 후처리 규칙으로 점수를 짜내는 대신 데이터 정의와 분할 단계에서 푼다.
- **그룹 분할 CV** (`GROUP_CV=True`): 변이 프로필이 같은 행(쌍둥이·중복)은 같은 fold에 묶는다. 같은 프로필이 학습/검증에 갈라져 OOF가 부풀려지는 것을 막는다. (`StratifiedGroupKFold`, 그룹 = 프로필 해시)
- **프로필 중복·라벨 충돌 진단 셀** 추가 (4-0절): 데이터 큐레이션의 근거.
- **쌍둥이 규칙 기본 OFF** (`USE_TWIN=False`): 분석가가 라벨 구조를 이용해 예측을 덮어쓰는 후처리라 확장성·규정 리스크가 크다. 진단용으로만 남겼고, 켜려면 플래그와 시뮬 이득 조건을 모두 만족해야 한다.
- (선택) `PROFILE_WEIGHT`: 같은 프로필 행이 학습을 지배하지 않도록 1/그룹크기 가중치. 기본 OFF, 효과는 CV로 확인 필요.
- 주의: 그룹 분할로 바뀌면 OOF 점수는 이전(무작위 분할)보다 낮아질 수 있다. 이전 점수가 중복 덕분에 부풀려져 있었다는 뜻이지 모델이 나빠진 것이 아니다.

**v9 변경점 (희소 변이 샘플 대응)**: 변이가 거의 없는 샘플이 다른 암종으로 오인되는 문제를 다룬다. 모두 CFG 플래그로 켜고 끌 수 있고, 4-2절 ablation으로 효과를 확인한 뒤 쓴다.
- `USE_SPARSE_META`: 변이 유전자 0개 / 2개 이하 / `TMB_LOW` 이하 희소성 지표 피처 (`sp_*`)
- `USE_KNN`: Jaccard 유사도 기반 KNN 피처 (`knn_p_*` = 가까운 train 환자들의 암종 분포, `knn_max_sim`, `knn_mean_sim`). **변이 프로필이 완전히 같은 행(자기 자신·쌍둥이·중복)은 이웃에서 제외**해서, 쌍둥이 라벨을 몰래 학습하는 우회로가 되지 않게 했다. 이웃은 항상 그 fold의 train 행에서만 찾는다 (valid/test도 동일).
- `SPARSE_WEIGHT`: 변이가 2개 이하인 학습 행의 가중치 배수 (1.0=사용 안 함)

**v10 변경점 (점수 + 재현성 + 현업성)**
- **쌍둥이 규칙 -> 학습형 중복 규칙(`USE_DUP_RULE`)**: 암종 이름을 코드에 박아 둔 `TWIN_SWAP` 대신, train에서 "같은 프로필을 가진 *다른* train 행들의 라벨 묶음 -> 내 라벨" 표를 leave-one-out으로 추정해 쓴다. 표본 수(`DUP_MIN_SUPPORT`)와 확률(`DUP_MIN_PROB`)이 기준 미만이면 규칙이 스스로 꺼지고, 어떤 암종 쌍에도 일반화된다. OOF 시뮬 이득이 `BOOST_MIN_GAIN` 미만이면 적용하지 않는다.
- **macroF1 클래스 보정(`USE_CLASS_SCALE`)**: 클래스별 확률 배율을 OOF로 맞춘다. 중첩 검증(다른 fold에서 맞추고 남은 fold로 평가)에서 이득이 확인될 때만 적용한다. test 분포 정보는 쓰지 않는다.
- **정직한 점수 보고(`HONEST_REPORT`)**: 주 CV는 무작위 분할(대회 조건과 같고 이전 점수와 비교 가능), 추가로 그룹 분할 점수를 따로 출력한다. 현업 기준 점수는 그룹 분할 쪽이다.
- **재현성 기록**: 실행 설정, 패키지 버전, 데이터 파일 해시, 점수, 적용된 후처리를 `run_manifest_*.json`으로 저장한다.
- **제출 파일 2종**: 규칙 없는 `_core`(현업형)와 규칙 적용 `_dup`을 함께 만든다. 어느 쪽을 낼지는 대회 규정 확인 후 선택.
- v9의 희소성/KNN 피처는 실데이터 검증 전이라 기본 OFF로 되돌렸다 (4-2절 ablation으로 확인 후 켜기).

**v11 변경점 (모델 다양성)**
- **LightGBM 추가(`USE_LGBM`)**: XGB와 분기 방식이 달라 오분류 패턴이 어긋나는 모델을 앙상블에 추가. 가중치는 (XGB, LR, LGBM) 격자(0.1 단위)에서 OOF macroF1로 고르되, **XGB+LR 대비 이득이 `BOOST_MIN_GAIN` 이상일 때만 3종 앙상블을 채택**한다 (잡음에 맞춘 가중치 방지).
- **4-2절 ablation을 그룹 분할(`FOLDS_GROUP`) 기준으로 변경**: 희소성/KNN 피처의 효과를 쌍둥이 중복에 흔들리지 않게 잰다.
- 클래스 가중치 추가 조정은 넣지 않았다: v10 실측에서 클래스 배율 보정의 중첩 검증 이득이 -0.0019로 나왔고, 가장 큰 오분류(KIPAN↔KIRC, GBMLGG↔LGG)는 프로필이 같아서 가중치로 풀리지 않는다.
- 참고: 조기종료(early stopping)를 검증 fold로 하므로 부스팅 모델의 OOF는 약간 낙관적일 수 있다 (XGB도 동일, 기존 방식 유지).

**v12 변경점 (생물학적 피처 4종, 전부 기본 OFF + 그룹 분할 ablation)**
- `USE_AA_PHYS`: 미스센스 변이의 물리화학적 변화량 요약 (Grantham 거리, 전하 변화, 소수성 변화, 부피 변화, 급진적/보수적 치환 수, 전하 반전 수). 400개 치환쌍 중 빈도가 낮아 버려지는 쌍의 정보를 저차원으로 모은다.
- `USE_HS_ALLELE`: `유전자|원래aa위치+바뀐aa` 토큰 (예: KRAS G12D vs G12C). 기존 `hs_` 토큰은 바뀐 아미노산을 구분하지 않는다.
- `USE_PW_TYPE`: 경로별로 **절단 변이(nonsense/frameshift/splice)가 있는 유전자 수**와 그 외 변이 유전자 수를 분리 (기능 상실 vs 그 외의 방향성 대리 지표).
- `USE_PW_PAIR`: 경로 쌍의 동반 변이 여부(66개). 트리는 상호작용을 스스로 찾지만 LR은 못 찾는다.
- 4-3절 ablation(`RUN_ABLATION3`)이 **그룹 분할 기준**으로 각각의 효과를 잰다. 이득이 확인된 것만 켠다.


In [1]:
# v11/v12에서 LightGBM이 미설치라 조용히 꺼져 있었다 -> 여기서 설치까지 시도한다
import importlib, subprocess, sys
for _pkg in ("lightgbm", "xgboost"):
    try:
        importlib.import_module(_pkg)
    except ImportError:
        print(f"{_pkg} 설치 중...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=False)


lightgbm 설치 중...


In [2]:
import os, re, warnings
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, MaxAbsScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
try:
    import lightgbm as lgb
except ImportError:
    lgb = None
warnings.filterwarnings("ignore")

CFG = dict(
    DATA_DIR=os.environ.get("DATA_DIR", "../../data"),
    SEED=42,
    N_FOLDS=int(os.environ.get("N_FOLDS", 5)),
    MIN_GENE_FREQ=5,        # train에서 변이 샘플이 이 값 미만인 유전자는 제외
    MIN_HOTSPOT_FREQ=5,     # gene|position 토큰 최소 빈도
    MIN_AA_FREQ=20,         # 아미노산 치환쌍 최소 빈도
    USE_AA_SUBS=True,       # 아미노산 치환 패턴 피처 on/off (CV로 효과 확인)
    USE_PATHWAYS=True,      # 경로 집계 피처 on/off
    USE_HOTSPOTS=True,      # gene|position, gene|trunc 토큰 on/off
    BALANCE=True,           # 클래스 불균형 보정 (sample weight / class_weight)
    XGB_N_EST=int(os.environ.get("XGB_N_EST", 2000)),
    LR_C=0.1,
    USE_COMBOS=True,        # 동반 변이 콤보 피처 on/off
    TMB_LOW=5,              # 극저변이 기준 (THCA/LAML 등)
    TMB_HIGH=30,            # 고변이 기준 (SKCM/UCEC/STES 등)
    BOOST_MIN_GAIN=0.003,   # 쌍둥이 규칙 등 후처리가 OOF macroF1을 이만큼 못 올리면 적용 안 함
    NORMALIZE_TOKENS=True,  # 변이 표기 통일(HGVS 구/신 표기) + 한 칸 안 중복 제거. 운영진 확인 전에는 False 제출본도 준비
    USE_DUP_RULE=True,      # 학습형 중복 규칙: 같은 프로필의 train 라벨 묶음 -> 라벨 표(LOO)로 덮어쓰기. 근거 부족/이득 없으면 자동 OFF
    DUP_MAX_CTX=3,          # 같은 프로필 행이 이 개수 이하인 맥락만 사용 (큰 중복 군집은 제외)
    DUP_MIN_SUPPORT=20,     # 표의 한 칸이 이 표본 수 이상이어야 사용
    DUP_MIN_PROB=0.90,      # 표의 한 칸에서 최다 라벨 비율이 이 이상이어야 사용
    USE_CLASS_SCALE=True,   # macroF1용 클래스별 확률 배율 (중첩 검증 이득이 BOOST_MIN_GAIN 이상일 때만 적용)
    SCALE_ITERS=3,          # 배율 좌표 탐색 반복 횟수
    HONEST_REPORT=True,     # 주 CV가 무작위 분할일 때, 그룹 분할 점수를 추가로 계산해 출력 (시간 약 2배)
    GROUP_CV=False,         # True면 주 CV를 그룹 분할로 (중복 규칙 시뮬 불가). 기본 False: 대회 조건과 같은 무작위 분할 + HONEST_REPORT로 그룹 점수 병기
    PROFILE_WEIGHT=False,   # 같은 프로필 행에 1/그룹크기 가중치 (중복 행이 학습을 지배하지 않게). CV로 확인 후 사용
    USE_SPARSE_META=False,   # 희소성 메타 피처(변이 유전자 0/<=2/<=TMB_LOW 지표)
    USE_KNN=False,          # Jaccard KNN 피처 (같은 프로필 행은 이웃에서 제외)
    KNN_K=15,               # 이웃 수
    SPARSE_WEIGHT=1.0,      # 변이 2개 이하 학습 행의 가중치 배수 (1.0=off, 예: 0.5)
    USE_LGBM=True,          # LightGBM을 앙상블에 추가 (XGB+LR 대비 이득이 BOOST_MIN_GAIN 미만이면 가중치 0)
    LGB_N_EST=int(os.environ.get("LGB_N_EST", 2000)),
    USE_AA_PHYS=False,      # 미스센스 물리화학적 변화량 요약 피처
    USE_HS_ALLELE=False,    # 유전자|원래aa+위치+바뀐aa 토큰 (KRAS G12D vs G12C 구분)
    USE_PW_TYPE=False,      # 경로별 절단 변이 유전자 수 / 그 외 변이 유전자 수
    USE_PW_PAIR=False,      # 경로 쌍 동반 변이 여부
    # ---------------- v13 ----------------
    SEEDS=[42, 7, 2024],    # XGB/LGBM 시드 평균 (빠르게 보려면 [42])
    USE_LR_NORM=True,       # 행(L2) 정규화 LR: TMB 차이가 로짓을 지배하지 않게
    LR_NORM_C=20.0,         # 행 정규화로 입력 크기가 작아져서 C를 크게 (LR_C=0.1은 정규화 안 한 LR용)
    USE_NB_STACK=False,     # 나이브베이즈 사후 로그확률(내부 그룹 OOF)을 XGB/LGBM 입력으로 (AUTO_ABLATION이 결정)
    USE_GENE_SPLIT=False,   # 유전자별 비침묵(gn_)/침묵(gs_) 변이 분리 이진 피처 (AUTO_ABLATION이 결정)
    USE_BLOSUM=False,       # BLOSUM62 치환 점수 요약 피처 (AUTO_ABLATION이 결정)
    NORM_HASH=True,         # 프로필 해시를 canon+정렬 토큰으로 만든다
    AUTO_ABLATION=True,     # 새/기존 생물학 피처 플래그를 그룹 분할 CV로 자동 선택 (시간 오래 걸림, 끄면 위 플래그 그대로)
    ABL_MIN_GAIN=0.003,     # 플래그 하나를 켰을 때 macroF1 이득이 이 값 이상이어야 채택
    ABL_FAST=True,          # ablation은 XGB(lr 0.15, 시드 1개) + 행정규화 LR로 빠르게
    USE_LABEL_SHIFT=True,   # 라벨 분포 이동(BBSE) 보정: 시뮬 이득이 있을 때만 _shift 제출본을 별도 생성
)
if CFG["USE_LGBM"] and lgb is None:
    print("lightgbm 미설치 -> USE_LGBM=False (pip install lightgbm)"); CFG["USE_LGBM"] = False


## 1. 데이터 로드

In [3]:
D = CFG["DATA_DIR"]
train = pd.read_csv(f"{D}/train.csv").fillna("WT")
test = pd.read_csv(f"{D}/test.csv").fillna("WT")
sub = pd.read_csv(f"{D}/sample_submission.csv")

assert (test["ID"].values == sub["ID"].values).all(), "test와 sample_submission의 ID 순서가 다릅니다"
gene_cols = [c for c in train.columns if c not in ("ID", "SUBCLASS")]
assert gene_cols == [c for c in test.columns if c != "ID"], "train/test 유전자 컬럼이 다릅니다"

le = LabelEncoder()
y = le.fit_transform(train["SUBCLASS"])
n_class = len(le.classes_)
print("train", train.shape, "classes", n_class)
print(train["SUBCLASS"].value_counts())

train (6201, 4386) classes 26
SUBCLASS
BRCA      786
KIPAN     515
GBMLGG    461
STES      379
KIRC      334
THCA      324
SKCM      276
PRAD      266
OV        253
LGG       229
HNSC      223
COAD      223
SARC      198
UCEC      198
LUAD      184
LUSC      178
LIHC      158
LAML      158
CESC      155
PCPG      147
TGCT      124
PAAD      120
BLCA      104
THYM       98
ACC        72
DLBC       38
Name: count, dtype: int64


In [4]:
train.head(5)

,ID,SUBCLASS,A2M,AAAS,AADAT,AARS1,ABAT,ABCA1,ABCA2,ABCA3,...,ZNF292,ZNF365,ZNF639,ZNF707,ZNFX1,ZNRF4,ZPBP,ZW10,ZWINT,ZYX
0,TRAIN_0000,KIPAN,WT,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
1,TRAIN_0001,SARC,WT,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
2,TRAIN_0002,SKCM,R895R,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
3,TRAIN_0003,KIRC,WT,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
4,TRAIN_0004,GBMLGG,WT,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT


## EDA

In [5]:
import pandas as pd
import numpy as np
import re

# 1. 데이터 로드
train = pd.read_csv(f"{D}/train.csv").fillna("WT")

print("="*50)
print("1. 기본 데이터 크기 및 암종(SUBCLASS) 개수 확인")
print("="*50)
print(f"Train 크기: {train.shape}")

# 암종(SUBCLASS) 개수 및 분포 확인
subclass_counts = train["SUBCLASS"].value_counts()
print(f"총 암종(Subclass) 개수: {len(subclass_counts)}개")
print("\n[상위 5개 암종 분포]")
print(subclass_counts.head())

# 유전자 컬럼 분리
gene_cols = [c for c in train.columns if c not in ("ID", "SUBCLASS")]
print(f"\n유전자(Feature) 컬럼 총 개수: {len(gene_cols)}개")


print("\n" + "="*50)
print("2. 샘플당 변이 유전자 수 분포 (TMB)")
print("="*50)
# WT가 아닌 유전자의 개수를 행별로 카운트
mut_matrix = (train[gene_cols] != "WT")
mut_counts_per_sample = mut_matrix.sum(axis=1)
print(mut_counts_per_sample.describe().round(2))


print("\n" + "="*50)
print("3. 한 셀에 변이가 여러 개 들어있는지 확인 (구분자 체크)")
print("="*50)
# WT가 아닌 셀들만 추출
all_vals = train[gene_cols].values.ravel()
non_wt_vals = all_vals[all_vals != "WT"]

non_wt_series = pd.Series(non_wt_vals)
# 쉼표(,), 콜론(:), 세미콜론(;), 파이프(|), 공백 등이 포함된 비율 확인
sep_pattern = r"[,:;|\s]"
has_sep = non_wt_series.str.contains(sep_pattern, regex=True).mean()
print(f"Non-WT 값 중 구분자(, : ; | 공백)가 포함된 비율: {has_sep:.4%}")


print("\n" + "="*50)
print("4. 변이 표기 패턴 종류 확인 (정규식 기반 상위 패턴)")
print("="*50)
# 숫자를 'N'으로 치환하여 대표적인 변이 표기 패턴 분석
pattern_samples = non_wt_series.str.replace(r"\d+", "N", regex=True)
print(pattern_samples.value_counts().head(15))


print("\n" + "="*50)
print("5. 유전자별 변이 빈도 (희소성 확인)")
print("="*50)
gene_mut_freq = mut_matrix.sum(axis=0)
print(f"변이가 단 1개라도 발생한 유전자 수: {(gene_mut_freq > 0).sum()} / {len(gene_cols)}")
print(f"대다수 유전자의 변이 빈도 요약 (상위/하위):")
print(gene_mut_freq.describe().round(2))

1. 기본 데이터 크기 및 암종(SUBCLASS) 개수 확인
Train 크기: (6201, 4386)
총 암종(Subclass) 개수: 26개

[상위 5개 암종 분포]
SUBCLASS
BRCA      786
KIPAN     515
GBMLGG    461
STES      379
KIRC      334
Name: count, dtype: int64

유전자(Feature) 컬럼 총 개수: 4384개

2. 샘플당 변이 유전자 수 분포 (TMB)
count    6201.00
mean       35.30
std       100.78
min         0.00
25%         7.00
50%        14.00
75%        27.00
max      2393.00
dtype: float64

3. 한 셀에 변이가 여러 개 들어있는지 확인 (구분자 체크)
Non-WT 값 중 구분자(, : ; | 공백)가 포함된 비율: 10.0625%

4. 변이 표기 패턴 종류 확인 (정규식 기반 상위 패턴)
LNL    7409
ENK    6790
SNS    4517
RNH    4441
RNC    4141
RNQ    4127
PNP    4037
ANA    3935
TNT    3841
ANT    3712
DNN    3685
INI    3680
ANV    3652
GNG    3581
PNS    3536
Name: count, dtype: int64

5. 유전자별 변이 빈도 (희소성 확인)
변이가 단 1개라도 발생한 유전자 수: 4230 / 4384
대다수 유전자의 변이 빈도 요약 (상위/하위):
count    4384.00
mean       49.93
std        57.64
min         0.00
25%        20.00
50%        36.00
75%        61.00
max      1770.00
dtype: float64


## 2. 변이 표기 형식 확인 (파싱 규칙이 맞는지 눈으로 검증)

In [6]:
sample = pd.Series(train[gene_cols].iloc[:1000].to_numpy(dtype=object).ravel())
nonwt = sample[sample != "WT"]
print(f"non-WT 비율: {len(nonwt)/len(sample):.4%}")
print("한 셀에 구분자(, : ; | 공백)가 있는 비율:", nonwt.str.contains(r"[,:;|\s]").mean().round(4))
print(nonwt.str.replace(r"\d+", "N", regex=True).value_counts().head(25))

non-WT 비율: 0.8337%
한 셀에 구분자(, : ; | 공백)가 있는 비율: 0.0873
LNL    1236
ENK    1125
SNS     777
RNH     752
ANA     738
RNC     724
PNP     711
RNQ     696
TNT     665
ANT     642
GNG     636
ANV     629
FNF     614
PNL     610
DNN     609
PNS     593
INI     561
VNV     555
RNR     492
RNW     483
SNF     437
RN*     410
EN*     388
GNR     341
KNN     328
Name: count, dtype: int64


## 3. 피처 생성

- `g_*` : 유전자별 변이 여부 (이진)
- TMB 계열 : 변이 유전자 수, log, 이벤트 수
- 변이 유형 : missense / silent / nonsense / frameshift / splice / inframe 개수와 비율
- `hs_*` : 핫스팟(`유전자|원래아미노산+위치`) 및 truncating(`유전자|trunc`) 이진
- `aa_*` : 아미노산 치환쌍 비율
- `pw_*` : 경로별 변이 유전자 수 (아래 `PATHWAYS` 사전은 초안이니 자유롭게 수정)

In [7]:
def combo_features(ex):
    mut = ex["mut"]; n = mut.shape[0]
    n_genes = mut.sum(1)
    low, high = n_genes <= CFG["TMB_LOW"], n_genes >= CFG["TMB_HIGH"]

    def G(name):   # 유전자 변이 여부 (없는 유전자는 0 벡터 — 엉뚱한 컬럼 안 가져옴)
        j = gene_pos.get(name)
        return mut[:, j].copy() if j is not None else np.zeros(n, bool)

    tok_rows = {}
    for r, nm in zip(ex["tok_rows"], ex["tok_names"]):
        tok_rows.setdefault(nm, []).append(r)
    def T(name):   # 핫스팟/절단형 토큰 보유 여부 (MIN_HOTSPOT_FREQ와 무관하게 직접 계산)
        v = np.zeros(n, bool)
        if name in tok_rows: v[tok_rows[name]] = True
        return v

    f = {}
    # COAD
    f["COAD_APCtr_TP53"]  = T("APC|trunc") & G("TP53")
    f["COAD_APCtr_KRAS"]  = T("APC|trunc") & (G("KRAS") | G("FBXW7"))
    f["COAD_score"]       = T("APC|trunc").astype(int) + G("TP53") + G("KRAS") + G("FBXW7") + G("PIK3CA")
    # KIRC / KIPAN
    f["KIRC_VHL_PBRM1"]   = G("VHL") & G("PBRM1")
    f["KIRC_3p_score"]    = G("VHL").astype(int) + G("PBRM1") + G("SETD2") + G("BAP1")
    # UCEC
    f["UCEC_PTEN_CTNNB1"] = G("PTEN") & G("CTNNB1")
    f["UCEC_PTEN_PIK3CA"] = G("PTEN") & G("PIK3CA")
    f["UCEC_highTMB_PTEN"]= high & G("PTEN")
    # PAAD
    f["PAAD_score"]       = T("KRAS|G12").astype(int) + G("TP53") + T("CDKN2A|trunc") + G("SMAD4")
    # LGG vs GBMLGG
    f["LGG_IDH1_TP53_ATRX"] = T("IDH1|R132") & G("TP53") & G("ATRX")
    f["LGG_IDH1_clean"]   = T("IDH1|R132") & ~(G("PTEN") | G("EGFR"))
    f["GBM_PTEN_EGFR"]    = G("PTEN") & G("EGFR")
    f["GBM_noIDH1_signal"]= (G("PTEN") | G("EGFR")) & ~G("IDH1")
    # BRAF V600 + TMB (THCA vs SKCM vs COAD)
    f["THCA_BRAF_lowTMB"] = T("BRAF|V600") & low
    f["SKCM_BRAF_highTMB"]= T("BRAF|V600") & high
    f["SKCM_NRAS_highTMB"]= T("NRAS|Q61") & high
    f["COAD_BRAF_APC"]    = T("BRAF|V600") & T("APC|trunc")
    # 폐암 / 위식도 / 간 / 난소 / 백혈병
    f["LUAD_EGFR_noTP53"] = G("EGFR") & ~G("TP53")
    f["LUSC_TP53_noEGFR_highTMB"] = G("TP53") & ~G("EGFR") & high
    f["STES_highTMB_TP53_noAPC"]  = high & G("TP53") & ~T("APC|trunc")
    f["LIHC_CTNNB1_noAPC"]= G("CTNNB1") & ~G("APC")
    f["OV_TP53_lowTMB"]   = G("TP53") & (n_genes <= 20)
    f["LAML_NPM1_lowTMB"] = G("NPM1") & low

    names = list(f.keys())
    return np.c_[[np.asarray(f[k], np.float32) for k in names]].T, names

In [8]:
AA = "ACDEFGHIKLMNPQRSTVWY"
AA_IDX = {a: i for i, a in enumerate(AA)}
TYPES = ["missense", "silent", "nonsense", "frameshift", "splice", "inframe", "other"]
TYPE_IDX = {t: i for i, t in enumerate(TYPES)}
SEP = re.compile(r"[,:;|\s]+")
AA3 = {"Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C","Gln":"Q","Glu":"E","Gly":"G","His":"H","Ile":"I",
       "Leu":"L","Lys":"K","Met":"M","Phe":"F","Pro":"P","Ser":"S","Thr":"T","Trp":"W","Tyr":"Y","Val":"V","Ter":"*"}

_A3 = "|".join(AA3)   # Ala|Arg|...|Ter
# 단순 치환형(예: Arg1450Ter, r1450x, ARG1450TER, p.(Arg1450Ter))은 통째로 매칭해서 대소문자까지 한 번에 통일
_SIMPLE = re.compile(rf"^({_A3}|[A-Z])(\d+)({_A3}|[A-Z]|\*)$", re.I)

def _aa1(x, last=False):
    """아미노산 표기 하나 -> 1글자. 마지막 자리의 X/x/Ter는 종결 코돈(*)으로 읽는다."""
    if x == "*":
        return x
    if len(x) == 3:
        return AA3.get(x.capitalize(), x)
    x = x.upper()
    return "*" if (last and x == "X") else x

def canon(tok):
    """변이 표기 1개를 한 가지 모양으로 통일한다 (그 토큰만 보고 바꿈 = 행 독립, train/test 동일 적용).
    p.Arg1450Ter / p.(Arg1450Ter) / R1450X / r1450x / R1450* -> R1450*   (옛 HGVS의 X도 종결 코돈으로 읽는 기존 동작 유지)
    v7 보강: 괄호(HGVS 예측 표기), 대문자 P., 소문자 x/ter, 대소문자 섞인 3글자 표기도 같은 변이로 통일."""
    t = tok.strip()
    t = re.sub(r"^\(?[pP]\.", "", t)      # p. / P. / (p. 접두 제거
    t = t.strip("()")                       # p.(Arg1450Ter) 같은 괄호 제거
    m = _SIMPLE.match(t)
    if m:
        a, pos, b = m.groups()
        a1, b1 = _aa1(a), _aa1(b, last=True)
        if a1 != "*":                       # 종결 코돈이 시작 자리에 오는 표기는 건드리지 않는다
            return f"{a1}{pos}{b1}"
    t = re.sub(r"[A-Z][a-z]{2}", lambda m: AA3.get(m.group(), m.group()), t)   # 3글자 아미노산 -> 1글자 (fs/del 등 복합 표기용)
    t = re.sub(r"^([A-Z])(\d+)[Xx]$", r"\1\2*", t)                             # R1450X -> R1450*
    return t

def cell_tokens(cell):
    """셀 하나 -> 변이 토큰 목록. NORMALIZE_TOKENS면 표기를 통일하고 같은 변이의 중복 표기를 하나로 합친다."""
    toks = [t for t in SEP.split(str(cell)) if t and t != "WT"]
    if CFG["NORMALIZE_TOKENS"]:
        toks = list(dict.fromkeys(canon(t) for t in toks))
    return toks
RE_MIS = re.compile(r"^(?:p\.)?([A-Z])(\d+)([A-Z])$")
RE_NON = re.compile(r"^(?:p\.)?[A-Z]\d+(\*|X|Ter)$")

def parse_token(tok):
    """변이 표기 1개 -> (유형, 원래aa, 위치, 바뀐aa). 표기가 다르면 여기 규칙을 수정하세요."""
    low = tok.lower()
    m = re.search(r"\d+", tok)
    pos = m.group() if m else ""
    if "fs" in low or "frameshift" in low:
        return "frameshift", "", pos, ""
    if "splice" in low:
        return "splice", "", pos, ""
    if "del" in low or "ins" in low or "dup" in low:
        return "inframe", "", pos, ""
    if RE_NON.match(tok):
        return "nonsense", "", pos, ""
    m = RE_MIS.match(tok)
    if m:
        fa, p, ta = m.groups()
        return ("silent" if fa == ta else "missense"), fa, p, ta
    return "other", "", pos, ""

# ---- 아미노산 물리화학 표 (Grantham 1974: 조성 c, 극성 p, 부피 v) ----
_CPV = {"S":(1.42,9.2,32),"R":(0.65,10.5,124),"L":(0,4.9,111),"P":(0.39,8.0,32.5),"T":(0.71,8.6,61),"A":(0,8.1,31),
        "V":(0,5.9,84),"G":(0.74,9.0,3),"I":(0,5.2,111),"F":(0,5.2,132),"Y":(0.20,6.2,136),"C":(2.75,5.5,55),
        "H":(0.58,10.4,96),"Q":(0.89,10.5,85),"N":(1.33,11.6,56),"K":(0.33,11.3,119),"D":(1.38,13.0,54),
        "E":(0.92,12.3,83),"M":(0,5.7,105),"W":(0.13,5.4,170)}
_KD = {"A":1.8,"R":-4.5,"N":-3.5,"D":-3.5,"C":2.5,"Q":-3.5,"E":-3.5,"G":-0.4,"H":-3.2,"I":4.5,
       "L":3.8,"K":-3.9,"M":1.9,"F":2.8,"P":-1.6,"S":-0.8,"T":-0.7,"W":-0.9,"Y":-1.3,"V":4.2}   # Kyte-Doolittle 소수성
_CH = {a: 0.0 for a in AA}; _CH.update(K=1.0, R=1.0, D=-1.0, E=-1.0, H=0.1)                          # 대략적인 전하
def _grantham(a, b):
    (c1, p1, v1), (c2, p2, v2) = _CPV[a], _CPV[b]
    return 50.723 * (1.833 * (c1 - c2) ** 2 + 0.1018 * (p1 - p2) ** 2 + 0.000399 * (v1 - v2) ** 2) ** 0.5
GR = np.array([[_grantham(a, b) for b in AA] for a in AA])
assert abs(GR[AA_IDX["S"], AA_IDX["R"]] - 110) < 2 and abs(GR[AA_IDX["C"], AA_IDX["W"]] - 215) < 2 \
       and abs(GR[AA_IDX["L"], AA_IDX["I"]] - 5) < 1 and np.allclose(GR, GR.T)      # 문헌에 알려진 대표값으로 확인
BLOSUM62 = np.array([
    [ 4, 0,-2,-1,-2, 0,-2,-1,-1,-1,-1,-2,-1,-1,-1, 1, 0, 0,-3,-2],
    [ 0, 9,-3,-4,-2,-3,-3,-1,-3,-1,-1,-3,-3,-3,-3,-1,-1,-1,-2,-2],
    [-2,-3, 6, 2,-3,-1,-1,-3,-1,-4,-3, 1,-1, 0,-2, 0,-1,-3,-4,-3],
    [-1,-4, 2, 5,-3,-2, 0,-3, 1,-3,-2, 0,-1, 2, 0, 0,-1,-2,-3,-2],
    [-2,-2,-3,-3, 6,-3,-1, 0,-3, 0, 0,-3,-4,-3,-3,-2,-2,-1, 1, 3],
    [ 0,-3,-1,-2,-3, 6,-2,-4,-2,-4,-3, 0,-2,-2,-2, 0,-2,-3,-2,-3],
    [-2,-3,-1, 0,-1,-2, 8,-3,-1,-3,-2, 1,-2, 0, 0,-1,-2,-3,-2, 2],
    [-1,-1,-3,-3, 0,-4,-3, 4,-3, 2, 1,-3,-3,-3,-3,-2,-1, 3,-3,-1],
    [-1,-3,-1, 1,-3,-2,-1,-3, 5,-2,-1, 0,-1, 1, 2, 0,-1,-2,-3,-2],
    [-1,-1,-4,-3, 0,-4,-3, 2,-2, 4, 2,-3,-3,-2,-2,-2,-1, 1,-2,-1],
    [-1,-1,-3,-2, 0,-3,-2, 1,-1, 2, 5,-2,-2, 0,-1,-1,-1, 1,-1,-1],
    [-2,-3, 1, 0,-3, 0, 1,-3, 0,-3,-2, 6,-2, 0, 0, 1, 0,-3,-4,-2],
    [-1,-3,-1,-1,-4,-2,-2,-3,-1,-3,-2,-2, 7,-1,-2,-1,-1,-2,-4,-3],
    [-1,-3, 0, 2,-3,-2, 0,-3, 1,-2, 0, 0,-1, 5, 1, 0,-1,-2,-2,-1],
    [-1,-3,-2, 0,-3,-2, 0,-3, 2,-2,-1, 0,-2, 1, 5,-1,-1,-3,-3,-2],
    [ 1,-1, 0, 0,-2, 0,-1,-2, 0,-2,-1, 1,-1, 0,-1, 4, 1,-2,-3,-2],
    [ 0,-1,-1,-1,-2,-2,-2,-1,-1,-1,-1, 0,-1,-1,-1, 1, 5, 0,-2,-2],
    [ 0,-1,-3,-2,-1,-3,-3, 3,-2, 1, 1,-3,-2,-2,-3,-2, 0, 4,-3,-1],
    [-3,-2,-4,-3, 1,-2,-2,-3,-3,-2,-1,-4,-4,-2,-3,-3,-2,-3,11, 2],
    [-2,-2,-3,-2, 3,-3, 2,-1,-2,-1,-1,-2,-3,-1,-2,-2,-2,-1, 2, 7],
])
assert BLOSUM62[AA_IDX["W"], AA_IDX["W"]] == 11 and BLOSUM62[AA_IDX["A"], AA_IDX["A"]] == 4 \
       and BLOSUM62[AA_IDX["R"], AA_IDX["K"]] == 2 and np.allclose(BLOSUM62, BLOSUM62.T)   # 문헌값 확인
PHYS_NAMES = ["grantham_mean", "dcharge_mean", "dhydro_mean", "dvol_mean", "n_radical", "n_conservative", "n_charge_rev"]

def extract(df):
    arr = df[gene_cols].to_numpy(dtype=object)
    mut = arr != "WT"
    n = len(df)
    type_cnt = np.zeros((n, len(TYPES)), np.float32)
    aa_cnt = np.zeros((n, 400), np.float32)
    phys = np.zeros((n, 8), np.float32)       # 물리화학 합계 4개 + 급진/보수/전하반전 수 + 미스센스 수
    trunc_mut = np.zeros(mut.shape, bool)      # 유전자별 절단 변이 여부
    silent_mut = np.zeros(mut.shape, bool)     # 유전자별 침묵 변이 여부 (v13)
    ns_mut = np.zeros(mut.shape, bool)         # 유전자별 비침묵 변이 여부 (v13)
    blo = np.zeros((n, 4), np.float32); blo[:, 1] = 99   # BLOSUM 합, 최소, 음수 개수, 미스센스 수 (v13)
    tok_rows, tok_names = [], []
    al_rows, al_names = [], []                 # 바뀐 아미노산까지 포함한 토큰
    rr, cc = np.nonzero(mut)
    for r, c in zip(rr, cc):
        gene = gene_cols[c]
        for tok in cell_tokens(arr[r, c]):
            typ, fa, pos, ta = parse_token(tok)
            type_cnt[r, TYPE_IDX[typ]] += 1
            if typ == "silent": silent_mut[r, c] = True
            else: ns_mut[r, c] = True
            if typ == "missense":
                tok_rows.append(r); tok_names.append(f"{gene}|{fa}{pos}")
                al_rows.append(r); al_names.append(f"{gene}|{fa}{pos}{ta}")
                if fa in AA_IDX and ta in AA_IDX:
                    aa_cnt[r, AA_IDX[fa] * 20 + AA_IDX[ta]] += 1
                    b_ = BLOSUM62[AA_IDX[fa], AA_IDX[ta]]
                    blo[r, 0] += b_; blo[r, 1] = min(blo[r, 1], b_); blo[r, 2] += b_ < 0; blo[r, 3] += 1
                    g_ = GR[AA_IDX[fa], AA_IDX[ta]]
                    phys[r] += [g_, abs(_CH[fa] - _CH[ta]), abs(_KD[fa] - _KD[ta]), abs(_CPV[fa][2] - _CPV[ta][2]) / 10,
                                g_ >= 150, g_ <= 50, _CH[fa] * _CH[ta] < 0, 1]
            elif typ in ("nonsense", "frameshift", "splice"):
                tok_rows.append(r); tok_names.append(f"{gene}|trunc"); trunc_mut[r, c] = True
            elif typ == "inframe":                       # ← 추가
                tok_rows.append(r); tok_names.append(f"{gene}|inframe")        # 유전자 단위
                tok_rows.append(r); tok_names.append(f"{gene}|{pos}_inframe")  # 위치 단위 (EGFR 746_inframe 등)
    blo[blo[:, 3] == 0, 1] = 0
    return dict(mut=mut, type_cnt=type_cnt, aa_cnt=aa_cnt, phys=phys, trunc_mut=trunc_mut,
                silent_mut=silent_mut, ns_mut=ns_mut, blo=blo,
                tok_rows=np.array(tok_rows, dtype=np.int64), tok_names=tok_names,
                al_rows=np.array(al_rows, dtype=np.int64), al_names=al_names)

ex_tr, ex_te = extract(train), extract(test)
print("파싱 완료. 유형별 총 개수(train):", dict(zip(TYPES, ex_tr["type_cnt"].sum(0).astype(int))))
# ---- 표기 통일 동작 확인 ----
assert canon("R1450X") == "R1450*" == canon("p.Arg1450Ter") == canon("p.R1450*")
assert canon("E746_A750del") == "E746_A750del" and canon("S221Kfs") == "S221Kfs"
assert parse_token(canon("R1450X"))[0] == "nonsense"
# v7 보강 케이스
assert canon("p.(Arg1450Ter)") == canon("R1450x") == canon("ARG1450TER") == canon("P.Arg1450X") == "R1450*"
assert canon("p.Val600Glu") == canon("(V600E)") == canon("v600e") == "V600E"
assert canon("p.Arg97ProfsTer23") == "R97Pfs*23"
assert canon("X123R") == "X123R" and canon("Ter123Arg") == "*123R"
assert canon("p.Arg1450Xaa") == "R1450Xaa"      # 미지정 아미노산은 종결로 읽지 않는다
_save = CFG["NORMALIZE_TOKENS"]; CFG["NORMALIZE_TOKENS"] = True
assert cell_tokens("R1450X R1450* p.Arg1450Ter") == ["R1450*"]          # 같은 변이의 여러 표기 -> 1개
assert cell_tokens("R175H, V600E") == ["R175H", "V600E"] and cell_tokens("WT") == []
CFG["NORMALIZE_TOKENS"] = _save


파싱 완료. 유형별 총 개수(train): {'missense': np.int64(161051), 'silent': np.int64(64844), 'nonsense': np.int64(13080), 'frameshift': np.int64(9767), 'splice': np.int64(0), 'inframe': np.int64(3), 'other': np.int64(319)}


In [9]:
# ---- 파서 점검: 'other'로 분류된 토큰이 실제로 어떤 모양인가 (train 전용) ----
_arr = train[gene_cols].to_numpy(dtype=object)
_rr, _cc = np.nonzero(_arr != "WT")
_oth, _tot = Counter(), 0
for r_, c_ in zip(_rr, _cc):
    for tok_ in cell_tokens(_arr[r_, c_]):
        _tot += 1
        if parse_token(tok_)[0] == "other":
            _oth[re.sub(r"\d+", "N", tok_)] += 1
print(f"전체 토큰 {_tot:,}개 중 other {sum(_oth.values()):,}개 ({sum(_oth.values()) / max(_tot, 1):.3%})")
print("other 모양 상위:", _oth.most_common(15))
print("-> 이 비율이 1% 미만이면 파서 세분화로 얻을 점수는 거의 없다.")


전체 토큰 249,064개 중 other 319개 (0.128%)
other 모양 상위: [('*N*', 89), ('N_NLL>LL', 12), ('N_NAL>AL', 8), ('N_NSL>SL', 7), ('N_NPL>PL', 7), ('N_NMG>I*', 7), ('N_NWE>*K', 5), ('N_NFL>FL', 4), ('N_NMA>IS', 4), ('N_NTL>TL', 4), ('N_NME>I*', 4), ('N_NME>IK', 4), ('N_NKE>N*', 4), ('N_NLE>F*', 3), ('N_NLG>F*', 3)]
-> 이 비율이 1% 미만이면 파서 세분화로 얻을 점수는 거의 없다.


### 표기 점검 (train 전용)
train 데이터의 변이 표기 현황을 확인한다. HGVS 표기 규약상 같은 변이가 여러 방식(`R1450X`, `R1450*`, `p.Arg1450Ter`)으로 적힐 수 있어, 입력 표기가 달라도 같은 변이로 취급하도록 정규화(`canon`)를 둔다.
정규화는 토큰 하나만 보고 바꾸는 행 독립 연산이며 train/test에 동일하게 적용된다.

**train 점검 결과**: 구식 종결 표기(`숫자+X`), `p.` 접두, 3글자 아미노산 표기는 train에 **0건**이다. 즉 이 정규화는 train 데이터로 필요성이 확인된 것이 아니라 HGVS 규약에 따른 방어적 처리이며, 적용 여부는 대회 운영진 확인 결과에 따른다 (`NORMALIZE_TOKENS` 플래그로 on/off, 제출 파일명에 norm/nonorm 표시).


In [10]:
# train에서 표기 혼재 여부를 확인한다 (test는 사용하지 않음)
def tokens_per_cell(df, normalize):
    """non-WT 셀 하나당 평균 변이 토큰 수 (normalize=True면 canon 적용 + 중복 제거)."""
    vals = df[gene_cols].to_numpy(dtype=object).ravel()
    vals = vals[vals != "WT"]
    n = [len(dict.fromkeys(canon(t) for t in SEP.split(str(v)) if t and t != "WT")) if normalize
         else len([t for t in SEP.split(str(v)) if t and t != "WT"]) for v in vals]
    return float(np.mean(n)) if n else 0.0

_vals = pd.Series(train[gene_cols].to_numpy(dtype=object).ravel())
_vals = _vals[_vals != "WT"]
_toks = pd.Series([t for v in _vals for t in SEP.split(str(v)) if t and t != "WT"])

x_end = _toks[_toks.str.contains(r"^(?:p\.)?[A-Z]\d+X$")]
print(f"train 전체 토큰 {len(_toks):,}개 중 '숫자+X' 종결 표기: {len(x_end)}개")
print(x_end.value_counts().head(10))
print("p. 접두 표기:", int(_toks.str.startswith("p.").sum()),
      "| 3글자 아미노산 표기:", int(_toks.str.contains(r"[A-Z][a-z]{2}\d|\d[A-Z][a-z]{2}").sum()))
print(f"train 칸당 변이 수 {tokens_per_cell(train, False):.2f} -> (통일·중복 제거 후) {tokens_per_cell(train, True):.2f}")


train 전체 토큰 255,164개 중 '숫자+X' 종결 표기: 0개
Series([], Name: count, dtype: int64)
p. 접두 표기: 0 | 3글자 아미노산 표기: 0
train 칸당 변이 수 1.17 -> (통일·중복 제거 후) 1.14


In [11]:
# ---- 경로/유전자군 사전 (초안: 필요에 맞게 수정/추가) ----
PATHWAYS = {
    "p53": ["TP53","MDM2","MDM4","CDKN2A","ATM","CHEK1","CHEK2"],
    "RTK_RAS": ["EGFR","ERBB2","ERBB3","ERBB4","FGFR1","FGFR2","FGFR3","FGFR4","MET","ALK","RET","KIT",
                "PDGFRA","KRAS","NRAS","HRAS","BRAF","RAF1","NF1","MAP2K1","MAP2K2","SOS1","PTPN11"],
    "PI3K": ["PIK3CA","PIK3CB","PIK3R1","PTEN","AKT1","AKT2","AKT3","MTOR","TSC1","TSC2","STK11","RICTOR","INPP4B"],
    "Wnt": ["APC","CTNNB1","AXIN1","AXIN2","RNF43","ZNRF3","TCF7L2","GSK3B","AMER1"],
    "CellCycle": ["RB1","CDK4","CDK6","CCND1","CCND2","CCND3","CCNE1","CDKN1A","CDKN1B","CDKN2B","E2F1","FBXW7"],
    "Notch": ["NOTCH1","NOTCH2","NOTCH3","NOTCH4","FBXW7","SPEN"],
    "Hippo": ["NF2","LATS1","LATS2","SAV1","YAP1","WWTR1","TEAD1"],
    "TGFb": ["TGFBR1","TGFBR2","SMAD2","SMAD3","SMAD4","ACVR1B","ACVR2A"],
    "NRF2": ["NFE2L2","KEAP1","CUL3"],
    "SWI_SNF": ["ARID1A","ARID1B","ARID2","SMARCA4","SMARCB1","PBRM1","SMARCA2"],
    "Chromatin": ["KMT2A","KMT2B","KMT2C","KMT2D","EZH2","SETD2","KDM6A","KDM5C","CREBBP","EP300",
                  "BAP1","ASXL1","DNMT3A","IDH1","IDH2","ATRX"],
    "DDR_MMR": ["BRCA1","BRCA2","PALB2","ATR","POLE","POLD1","MLH1","MSH2","MSH6","PMS2",
                "RAD51","RAD50","MRE11","FANCA","FANCD2","ERCC2"],
    "Splicing": ["SF3B1","SRSF2","U2AF1","RBM10"],
    "LongGenes": ["TTN","MUC16","MUC4","SYNE1","OBSCN","RYR2","RYR1","CSMD1","CSMD3","LRP1B","USH2A","FLG","PCLO"],
}
gene_pos = {g: i for i, g in enumerate(gene_cols)}
PW_IDX = {k: [gene_pos[g] for g in v if g in gene_pos] for k, v in PATHWAYS.items()}
print({k: f"{len(v)}/{len(PATHWAYS[k])}" for k, v in PW_IDX.items()})   # 데이터에 존재하는 유전자 수/사전 크기

{'p53': '5/7', 'RTK_RAS': '14/23', 'PI3K': '11/13', 'Wnt': '6/9', 'CellCycle': '12/12', 'Notch': '5/6', 'Hippo': '4/7', 'TGFb': '6/7', 'NRF2': '2/3', 'SWI_SNF': '0/7', 'Chromatin': '6/16', 'DDR_MMR': '11/16', 'Splicing': '2/4', 'LongGenes': '4/13'}


In [12]:
# ---- 유전자/토큰/아미노산쌍 사전은 "학습에 쓰는 행"에서만 만든다 (Data Leakage / CV 왜곡 방지) ----
# CV에서는 fold의 train 행만으로 사전을 만들고, 그 fold의 valid/test는 같은 사전으로 변환만 한다.
# test에만 있는 토큰은 tok_idx.get()이 None이라 무시된다.
def uniq_pairs(ex, in_rows):
    return set((r, nm) for r, nm in zip(ex["tok_rows"].tolist(), ex["tok_names"]) if in_rows[r])

def fit_vocab(ex, rows):
    rows = np.asarray(rows)
    in_rows = np.zeros(ex["mut"].shape[0], bool); in_rows[rows] = True
    keep_idx = np.where(ex["mut"][rows].sum(0) >= CFG["MIN_GENE_FREQ"])[0]
    cnt = Counter(nm for _, nm in uniq_pairs(ex, in_rows))
    keep_tokens = sorted(t for t, c in cnt.items() if c >= CFG["MIN_HOTSPOT_FREQ"])
    aa_keep = np.where(ex["aa_cnt"][rows].sum(0) >= CFG["MIN_AA_FREQ"])[0]
    al_keep = []
    if CFG["USE_HS_ALLELE"]:                       # 대립유전자 토큰도 fold의 train 행에서만 사전 구성
        cnt_a = Counter(nm for r_, nm in set(zip(ex["al_rows"].tolist(), ex["al_names"])) if in_rows[r_])
        al_keep = sorted(t for t, c in cnt_a.items() if c >= CFG["MIN_HOTSPOT_FREQ"])
    keep_ns = np.where(ex["ns_mut"][rows].sum(0) >= CFG["MIN_GENE_FREQ"])[0]       # v13: 비침묵/침묵 분리용
    keep_sl = np.where(ex["silent_mut"][rows].sum(0) >= CFG["MIN_GENE_FREQ"])[0]
    return dict(keep_idx=keep_idx, keep_ns=keep_ns, keep_sl=keep_sl, keep_tokens=keep_tokens,
                tok_idx={t: i for i, t in enumerate(keep_tokens)}, aa_keep=aa_keep,
                al_keep=al_keep, al_idx={t: i for i, t in enumerate(al_keep)})

def compose(ex, voc):
    mut = ex["mut"]; n = mut.shape[0]
    blocks, names = [], []
    def add(a, nm):
        blocks.append(np.asarray(a, np.float32).reshape(n, -1)); names.extend(nm)

    add(mut[:, voc["keep_idx"]], [f"g_{gene_cols[i]}" for i in voc["keep_idx"]])

    if CFG["USE_GENE_SPLIT"]:     # 비침묵(선택압 신호) / 침묵(배경 변이율 신호)을 따로
        if len(voc["keep_ns"]): add(ex["ns_mut"][:, voc["keep_ns"]], [f"gn_{gene_cols[i]}" for i in voc["keep_ns"]])
        if len(voc["keep_sl"]): add(ex["silent_mut"][:, voc["keep_sl"]], [f"gs_{gene_cols[i]}" for i in voc["keep_sl"]])
    n_genes = mut.sum(1)
    tc = ex["type_cnt"]; n_ev = tc.sum(1)
    add(np.c_[n_genes, np.log1p(n_genes), n_ev], ["n_mut_genes", "log_n_mut_genes", "n_events"])
    if CFG["USE_SPARSE_META"]:   # 정보량이 극히 적은 샘플 표시 (모델이 과신하지 않도록)
        add(np.c_[n_genes == 0, n_genes <= 2, n_genes <= CFG["TMB_LOW"]], ["sp_tmb0", "sp_tmb_le2", "sp_tmb_le_low"])
    add(tc, [f"cnt_{t}" for t in TYPES])
    trunc = tc[:, [TYPE_IDX[t] for t in ("nonsense", "frameshift", "splice")]].sum(1)
    add(np.c_[tc[:, TYPE_IDX["silent"]] / (n_ev + 1),
              tc[:, TYPE_IDX["missense"]] / (n_ev + 1),
              trunc / (n_ev + 1)], ["ratio_silent", "ratio_missense", "ratio_trunc"])

    if CFG["USE_HOTSPOTS"] and voc["keep_tokens"]:
        H = np.zeros((n, len(voc["keep_tokens"])), np.float32)
        for r, nm in zip(ex["tok_rows"], ex["tok_names"]):
            j = voc["tok_idx"].get(nm)
            if j is not None:
                H[r, j] = 1
        add(H, [f"hs_{t}" for t in voc["keep_tokens"]])
    if CFG["USE_HS_ALLELE"] and voc["al_keep"]:
        HA = np.zeros((n, len(voc["al_keep"])), np.float32)
        for r, nm in zip(ex["al_rows"], ex["al_names"]):
            j = voc["al_idx"].get(nm)
            if j is not None:
                HA[r, j] = 1
        add(HA, [f"ha_{t}" for t in voc["al_keep"]])
    if CFG["USE_BLOSUM"]:
        b = ex["blo"]; cnt_b = np.maximum(b[:, 3], 1)
        add(np.c_[b[:, 0] / cnt_b, b[:, 1], b[:, 2], b[:, 2] / cnt_b], ["aa_blosum_mean", "aa_blosum_min", "aa_blosum_nneg", "aa_blosum_fneg"])
    if CFG["USE_AA_PHYS"]:
        ph = ex["phys"]; cnt = np.maximum(ph[:, 7], 1)[:, None]
        add(np.c_[ph[:, :4] / cnt, ph[:, 4:7]], PHYS_NAMES)
    if CFG["USE_COMBOS"]:
        C, cn = combo_features(ex)
        add(C, [f"cb_{k}" for k in cn])
    if CFG["USE_AA_SUBS"] and len(voc["aa_keep"]):
        A = ex["aa_cnt"][:, voc["aa_keep"]] / (tc[:, TYPE_IDX["missense"]] + 1)[:, None]
        add(A, [f"aa_{AA[k // 20]}>{AA[k % 20]}" for k in voc["aa_keep"]])

    if CFG["USE_PATHWAYS"]:
        P = np.c_[[mut[:, idx].sum(1) if idx else np.zeros(n) for idx in PW_IDX.values()]].T
        add(P, [f"pw_{k}" for k in PW_IDX])
        if CFG["USE_PW_TYPE"]:        # 절단 변이가 있는 유전자 수 / 그 외 변이 유전자 수
            T = np.c_[[ex["trunc_mut"][:, idx].sum(1) if idx else np.zeros(n) for idx in PW_IDX.values()]].T
            add(T, [f"pwt_{k}_trunc" for k in PW_IDX]); add(np.maximum(P - T, 0), [f"pwt_{k}_other" for k in PW_IDX])
        if CFG["USE_PW_PAIR"]:        # 경로 쌍 동반 변이 (LongGenes는 신호경로가 아니므로 제외)
            ks = [k for k in PW_IDX if k != "LongGenes"]; kk = {k: j for j, k in enumerate(PW_IDX)}
            pairs = [(a, b) for x, a in enumerate(ks) for b in ks[x + 1:]]
            add(np.c_[[(P[:, kk[a]] > 0) & (P[:, kk[b]] > 0) for a, b in pairs]].T, [f"pwx_{a}__{b}" for a, b in pairs])

    return np.hstack(blocks), names

# 참고용(진단): 전체 train 기준 사전 크기. 모델 학습에는 쓰지 않고 fold마다 fit_vocab을 다시 호출한다.
_v = fit_vocab(ex_tr, np.arange(len(train)))
print(f"(참고) 전체 train 기준 유전자 {len(_v['keep_idx'])}/{len(gene_cols)}, 핫스팟 토큰 {len(_v['keep_tokens'])}, 아미노산쌍 {len(_v['aa_keep'])}")


(참고) 전체 train 기준 유전자 4171/4384, 핫스팟 토큰 1795, 아미노산쌍 158


### 4-0. 프로필 중복·라벨 충돌 진단 + 그룹 분할 준비

같은 환자가 개별 코호트(`KIRC`)와 통합 코호트(`KIPAN`)에 모두 등록되면 변이 프로필이 완전히 같은 행이 생긴다 (`LGG`↔`GBMLGG`도 동일). 이를 **후처리로 덮어쓰지 않고**, 먼저 규모를 진단하고 분할 단계에서 같은 프로필이 학습/검증에 갈라지지 않게 묶는다.
- 그룹 = 변이 프로필 해시. `StratifiedGroupKFold`로 클래스 비율은 유지하면서 같은 그룹은 같은 fold로 보낸다.
- 변이가 거의 없는 프로필(예: 전부 WT)은 우연히 같을 수 있으므로, 진단에서 변이 유전자 수 기준(`TMB_LOW`)으로 따로 센다.


In [13]:
import hashlib

def profile_hashes(df):
    """행마다 변이 프로필의 지문. NORM_HASH면 셀마다 canon 적용 + 토큰 정렬 후 해시 (표기/순서가 달라도 같은 프로필이면 같은 지문).
    변이가 있는 칸만 훑으므로 빠르다."""
    arr = df[gene_cols].to_numpy(dtype=object)
    out_h = []
    for r in arr:
        if CFG["NORM_HASH"]:
            parts = [f"{j}:" + ",".join(sorted(dict.fromkeys(canon(t) for t in SEP.split(str(r[j])) if t and t != "WT")))
                     for j in np.nonzero(r != "WT")[0]]
        else:
            parts = [f"{j}:{r[j]}" for j in np.nonzero(r != "WT")[0]]
        out_h.append(hashlib.md5("|".join(parts).encode()).hexdigest()[:12])
    return np.array(out_h)

H_tr, H_te = profile_hashes(train), profile_hashes(test)
lab_tr = train["SUBCLASS"].values

# ---- 진단: 같은 프로필 그룹과 라벨 충돌 ----
SIBLINGS = [frozenset(("KIRC", "KIPAN")), frozenset(("LGG", "GBMLGG"))]
g = pd.DataFrame({"h": H_tr, "y": lab_tr, "tmb": (train[gene_cols] != "WT").sum(1).values})
grp = g.groupby("h").agg(n=("y", "size"), labels=("y", lambda s: frozenset(s)), tmb=("tmb", "first"))
dup = grp[grp["n"] > 1]
conf = dup[dup["labels"].map(len) > 1]
sib = conf[conf["labels"].isin(SIBLINGS)]
print(f"train 프로필 그룹 {len(grp):,}개 중 2행 이상 중복: {len(dup):,}개 (해당 행 {int(dup['n'].sum()):,}개, 전체의 {dup['n'].sum()/len(train):.1%})")
print(f"  - 변이 유전자 {CFG['TMB_LOW']}개 이상인 중복 그룹(우연일 가능성 낮음): {int((dup['tmb'] >= CFG['TMB_LOW']).sum())}개")
print(f"  - 서로 다른 라벨이 섞인 그룹: {len(conf)}개 -> 형제 관계(KIRC/KIPAN, LGG/GBMLGG): {len(sib)}개, 그 외: {len(conf) - len(sib)}개")
if len(conf) - len(sib):
    print("  - 형제 외 라벨 충돌 예:", [tuple(sorted(s)) for s in conf[~conf['labels'].isin(SIBLINGS)]['labels'].head(5)])
in_tr = pd.Series(H_te).isin(set(H_tr))
print(f"test 행 중 train과 프로필이 같은 행: {int(in_tr.sum())}개 ({in_tr.mean():.1%})  <- 학습 데이터에 같은 프로필이 있다는 뜻 (정답 힌트로 쓰지 않음)")

# ---- 분할: 같은 프로필은 같은 fold로 ----
GRP_SIZE = pd.Series(H_tr).map(pd.Series(H_tr).value_counts()).to_numpy(dtype=float)   # 행마다 같은 프로필 행 수

def make_folds(group):
    if group:
        sp = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
        return list(sp.split(np.zeros(len(y)), y, groups=H_tr))
    sp = StratifiedKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
    return list(sp.split(np.zeros(len(y)), y))

FOLDS_RANDOM, FOLDS_GROUP = make_folds(False), make_folds(True)
assert all(not (set(H_tr[tr]) & set(H_tr[va])) for tr, va in FOLDS_GROUP), "그룹 분할 실패: 같은 프로필이 train/valid에 걸쳐 있음"
FOLDS = FOLDS_GROUP if CFG["GROUP_CV"] else FOLDS_RANDOM
print("주 CV 분할:", "그룹(프로필) 분할" if CFG["GROUP_CV"] else "무작위 분할", "| fold 크기:", [len(va) for _, va in FOLDS])


train 프로필 그룹 5,636개 중 2행 이상 중복: 451개 (해당 행 1,016개, 전체의 16.4%)
  - 변이 유전자 5개 이상인 중복 그룹(우연일 가능성 낮음): 394개
  - 서로 다른 라벨이 섞인 그룹: 447개 -> 형제 관계(KIRC/KIPAN, LGG/GBMLGG): 445개, 그 외: 2개
  - 형제 외 라벨 충돌 예: [('LGG', 'THCA'), ('BRCA', 'CESC', 'GBMLGG', 'KIPAN', 'KIRC', 'LAML', 'LGG', 'LUSC', 'PAAD', 'PCPG', 'PRAD', 'SARC', 'STES', 'THCA', 'THYM')]
test 행 중 train과 프로필이 같은 행: 266개 (10.4%)  <- 학습 데이터에 같은 프로필이 있다는 뜻 (정답 힌트로 쓰지 않음)
주 CV 분할: 무작위 분할 | fold 크기: [1241, 1240, 1240, 1240, 1240]


### 4-1. KNN 유사도 피처 (희소 변이 샘플 보강)

변이가 적은 샘플은 트리 모델이 분기 조건을 잡기 어렵다. 그래서 변이 유전자 집합의 **Jaccard 유사도**로 가장 가까운 train 환자 `KNN_K`명을 찾아 그들의 암종 분포를 피처로 준다.
- 이웃은 fold의 train 행에서만 찾는다. train 행 자신의 피처는 자기를 뺀 나머지에서, valid/test 행은 같은 train 행들에서 계산한다 (stacking의 leave-one-out 방식).
- **같은 프로필(해시 동일) 행은 이웃에서 뺀다.** 안 빼면 쌍둥이의 짝 라벨이 최근접 이웃으로 들어와 모델이 그 패턴을 몰래 학습한다(= 쌍둥이 규칙을 모델 안에 숨기는 셈). 빼면 "비슷하지만 같지는 않은" 환자의 분포만 쓴다.


In [14]:
from scipy import sparse

_ids, _ = pd.factorize(np.concatenate([H_tr, H_te]))      # 프로필 해시 -> 정수 id (train/test 공통)
ID_TR, ID_TE = _ids[:len(H_tr)], _ids[len(H_tr):]
KNN_NAMES = [f"knn_p_{c}" for c in le.classes_] + ["knn_max_sim", "knn_mean_sim"]

def knn_features(Q, q_id, R, r_id, r_y, k, chunk=512):
    """Q 행마다 R(참조 집합)에서 Jaccard 유사도 상위 k명의 암종 분포(유사도 가중) + 최대/평균 유사도.
    q_id == r_id 인 참조 행(자기 자신, 같은 프로필의 쌍둥이/중복)은 이웃에서 제외."""
    Qs = sparse.csr_matrix(np.asarray(Q) > 0, dtype=np.float32)
    Rs = sparse.csr_matrix(np.asarray(R) > 0, dtype=np.float32)
    nq = np.asarray(Qs.sum(1)).ravel(); nr = np.asarray(Rs.sum(1)).ravel()
    k = max(1, min(k, Rs.shape[0] - 1))
    out = np.zeros((Qs.shape[0], n_class + 2), np.float32)
    for s in range(0, Qs.shape[0], chunk):
        e = min(s + chunk, Qs.shape[0])
        inter = (Qs[s:e] @ Rs.T).toarray()
        union = nq[s:e, None] + nr[None, :] - inter
        sim = np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)
        sim[q_id[s:e, None] == r_id[None, :]] = -1.0
        idx = np.argpartition(-sim, k - 1, axis=1)[:, :k]
        ss = np.take_along_axis(sim, idx, 1).clip(min=0)
        lab = r_y[idx]
        for c in range(n_class):
            out[s:e, c] = (ss * (lab == c)).sum(1)
        tot = ss.sum(1, keepdims=True)
        out[s:e, :n_class] /= np.where(tot > 0, tot, 1.0)
        out[s:e, n_class] = ss.max(1); out[s:e, n_class + 1] = ss.mean(1)
    return out

# ---- 동작 확인 (작은 예제) ----
_n_class_bak = n_class; n_class = 3
_Rm = np.array([[1,1,0,0],[1,1,0,0],[1,0,1,0],[0,0,1,1],[0,0,0,0]], bool)   # 0번과 1번은 같은 프로필(쌍둥이)
_ry = np.array([0, 1, 0, 2, 2]); _rid = np.array([0, 0, 1, 2, 3])
_o = knn_features(_Rm, _rid, _Rm, _rid, _ry, k=2)
assert np.allclose(_o[0, :3], [1, 0, 0])      # 0번의 이웃: 같은 프로필인 1번(라벨1)은 제외 -> 2번(라벨0)이 가장 가까움
assert np.allclose(_o[1, :3], [1, 0, 0])      # 1번도 마찬가지 (쌍둥이 라벨이 새어 들어오지 않음)
assert np.allclose(_o[4, :3], 0)              # 변이 없는 행은 유사도 0 -> 분포 0 (가짜 확신 없음)
n_class = _n_class_bak
print("KNN 피처 동작 확인 통과, 피처 수:", len(KNN_NAMES))


KNN 피처 동작 확인 통과, 피처 수: 28


### 4-1b. 적대적 검증: train과 test를 구분할 수 있는가 (라벨 미사용)
test 예측 분포(COAD 548행 등)가 train 비율과 크게 다르다. 원인이 (a) **라벨 비율 자체가 다름**(prior shift)인지, (b) **피처 분포가 다름**(covariate shift)인지 가늠한다.
train/test를 구분하는 분류기의 AUC가 0.5 근처면 피처 분포는 같고, 0.6 이상이면 피처 분포가 다르다. 구분에 쓰인 피처 상위도 출력한다.


In [15]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

_va = fit_vocab(ex_tr, np.arange(len(train)))                 # 라벨 없이 train 전체 기준 사전 (진단 전용)
_Xa, _na = compose(ex_tr, _va); _Xb, _ = compose(ex_te, _va)
_Xadv = np.vstack([_Xa, _Xb]); _yadv = np.r_[np.zeros(len(_Xa)), np.ones(len(_Xb))]
_clf = xgb.XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=4, subsample=0.8, colsample_bytree=0.3,
                         tree_method="hist", n_jobs=-1, random_state=0)
_p = cross_val_predict(_clf, _Xadv, _yadv, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
print(f"적대적 검증 AUC = {roc_auc_score(_yadv, _p):.3f}   (0.5 근처: 피처 분포 같음 / 0.6+: 피처 분포 이동)")
_clf.fit(_Xadv, _yadv)
print("train/test 구분에 쓰인 피처 상위 10:", [_na[i] for i in np.argsort(-_clf.feature_importances_)[:10]])
_tm_tr, _tm_te = ex_tr["mut"].sum(1), ex_te["mut"].sum(1)
print("변이 유전자 수 분위(10/50/90%)  train:", np.percentile(_tm_tr, [10, 50, 90]), " test:", np.percentile(_tm_te, [10, 50, 90]))
del _Xa, _Xb, _Xadv


적대적 검증 AUC = 0.726   (0.5 근처: 피처 분포 같음 / 0.6+: 피처 분포 이동)
train/test 구분에 쓰인 피처 상위 10: ['hs_APC|trunc', 'cnt_inframe', 'g_AHNAK', 'g_TCHH', 'aa_T>P', 'cb_COAD_APCtr_TP53', 'aa_H>P', 'g_MTCH2', 'aa_N>T', 'g_CPEB2']
변이 유전자 수 분위(10/50/90%)  train: [ 3. 14. 67.]  test: [ 5. 28. 93.]


## 4. 5-fold CV (XGBoost + LightGBM + Logistic Regression ×2 + 선택적 NB 스태킹)

※ 시드 평균(`SEEDS`)과 행정규화 LR(`lrn`)이 v13에서 추가됐다. 부스팅은 검증 fold로 early stopping을 하므로 OOF가 약간 낙관적이다.
※ `USE_NB_STACK`: 나이브베이즈 사후확률 피처는 fold의 train 행에 대해 **내부 그룹 분할 OOF**로 만든다 (같은 프로필끼리 묶음). valid/test 행은 fold의 train 전체로 적합한 NB로 계산한다. 정답 라벨이 피처로 새지 않는다.


In [16]:
from sklearn.preprocessing import Normalizer
from sklearn.naive_bayes import MultinomialNB

MODELS = ("xgb", "lr", "lrn", "lgb")

def macro_f1(yt, pred, K):
    tp = np.bincount(yt[pred == yt], minlength=K).astype(float)
    pc, tc = np.bincount(pred, minlength=K), np.bincount(yt, minlength=K)
    d = pc + tc
    return float(np.mean(np.where(d > 0, 2 * tp / np.maximum(d, 1), 0.0)[d > 0]))

def active_models():
    return tuple(m for m in MODELS if (m != "lrn" or CFG["USE_LR_NORM"]) and (m != "lgb" or CFG["USE_LGBM"]))

# ---- 나이브베이즈 스태킹 ----
def _bin_cols(names):
    return np.array([i for i, n in enumerate(names) if n.startswith(("g_", "hs_", "ha_"))], dtype=int)

def _nb_post(nb, Xb):
    lp = nb.predict_log_proba(Xb)
    o = np.full((Xb.shape[0], n_class), -30.0, np.float32)
    o[:, nb.classes_] = np.clip(lp, -30, 0)        # fold에 없는 클래스는 -30
    return o

def nb_stack_features(Xb_tr, y_tr, g_tr, others, seed):
    """train 행: 내부 그룹 OOF, others(valid/test): train 전체로 적합한 NB. 출력 = [사후 로그확률, TMB 보정 사후 로그확률]."""
    def feats(lp, Xb):
        sc = (lp - lp.max(1, keepdims=True)) / (1.0 + np.sqrt(Xb.sum(1, keepdims=True)))
        return np.hstack([lp, sc]).astype(np.float32)
    oof = np.zeros((Xb_tr.shape[0], n_class), np.float32)
    for a, b in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed).split(Xb_tr, y_tr, groups=g_tr):
        oof[b] = _nb_post(MultinomialNB(alpha=0.3, fit_prior=False).fit(Xb_tr[a], y_tr[a]), Xb_tr[b])
    nb = MultinomialNB(alpha=0.3, fit_prior=False).fit(Xb_tr, y_tr)
    return feats(oof, Xb_tr), [feats(_nb_post(nb, Z), Z) for Z in others]
NB_NAMES = [f"nb_lp_{c}" for c in le.classes_] + [f"nb_sc_{c}" for c in le.classes_]

def run_cv(ex_tr, ex_te, verbose=True, folds=None, models=None, fast=False):
    """반환: R = {모델: (oof, test)}, imp = XGB 피처 중요도. fast=True면 ablation용 (XGB lr 0.15, 시드 1개)."""
    folds = FOLDS if folds is None else folds
    models = active_models() if models is None else tuple(models)
    n_te = ex_te["mut"].shape[0]
    R = {m: (np.zeros((len(y), n_class)), np.zeros((n_te, n_class))) for m in models}
    imp = Counter()
    seeds = CFG["SEEDS"][:1] if fast else CFG["SEEDS"]
    nf = len(folds)

    for fold, (tr, va) in enumerate(folds):
        voc = fit_vocab(ex_tr, tr)                      # 사전은 이 fold의 train 행만으로
        X_all, names = compose(ex_tr, voc)
        X_te, _ = compose(ex_te, voc)
        if CFG["USE_KNN"]:
            cols = voc["keep_idx"]
            Rm, r_id, r_y = ex_tr["mut"][tr][:, cols], ID_TR[tr], y[tr]
            X_all = np.hstack([X_all, knn_features(ex_tr["mut"][:, cols], ID_TR, Rm, r_id, r_y, CFG["KNN_K"])])
            X_te = np.hstack([X_te, knn_features(ex_te["mut"][:, cols], ID_TE, Rm, r_id, r_y, CFG["KNN_K"])])
            names = names + KNN_NAMES
        assert len(np.unique(y[tr])) == n_class, "일부 클래스가 train fold에 없습니다 (N_FOLDS를 줄이세요)"

        sw = compute_sample_weight("balanced", y[tr]) if CFG["BALANCE"] else None
        pw = None
        if CFG["PROFILE_WEIGHT"]:
            pw = 1.0 / GRP_SIZE[tr]; pw = pw / pw.mean()
        if CFG["SPARSE_WEIGHT"] != 1.0:
            lw = np.where(ex_tr["mut"][tr].sum(1) <= 2, CFG["SPARSE_WEIGHT"], 1.0)
            pw = lw if pw is None else pw * lw
        if pw is not None:
            sw = pw if sw is None else sw * pw

        # 트리 모델 입력 (+ NB 스태킹)
        Xt, Xt_te, tnames = X_all, X_te, names
        if CFG["USE_NB_STACK"] and ("xgb" in models or "lgb" in models):
            bc = _bin_cols(names)
            f_tr, (f_va, f_te) = nb_stack_features(X_all[tr][:, bc], y[tr], H_tr[tr], [X_all[va][:, bc], X_te[:, bc]], CFG["SEED"] + fold)
            nbX = np.zeros((len(y), 2 * n_class), np.float32); nbX[tr] = f_tr; nbX[va] = f_va
            Xt, Xt_te, tnames = np.hstack([X_all, nbX]), np.hstack([X_te, f_te]), names + NB_NAMES

        msg = f"fold{fold}"
        f1v = lambda p: f1_score(y[va], p[va].argmax(1), average="macro")

        if "xgb" in models:
            po, pt = 0.0, 0.0
            for k, sd in enumerate(seeds):
                m = xgb.XGBClassifier(
                    objective="multi:softprob", n_estimators=(600 if fast else CFG["XGB_N_EST"]),
                    learning_rate=(0.15 if fast else 0.08), max_depth=5, subsample=0.8, colsample_bytree=0.3,
                    tree_method="hist", eval_metric="mlogloss", early_stopping_rounds=(30 if fast else 50),
                    random_state=sd, n_jobs=-1)
                m.fit(Xt[tr], y[tr], sample_weight=sw, eval_set=[(Xt[va], y[va])], verbose=False)
                po = po + m.predict_proba(Xt[va]) / len(seeds); pt = pt + m.predict_proba(Xt_te) / len(seeds)
                if k == 0:
                    for nm_, v_ in zip(tnames, m.feature_importances_): imp[nm_] += v_ / nf
            R["xgb"][0][va] = po; R["xgb"][1][:] += pt / nf
            msg += f" | XGB macroF1={f1v(R['xgb'][0]):.4f}"

        if "lr" in models:
            lr = make_pipeline(MaxAbsScaler(), LogisticRegression(C=CFG["LR_C"], max_iter=500, class_weight="balanced" if CFG["BALANCE"] else None))
            lr.fit(X_all[tr], y[tr], **({} if pw is None else {"logisticregression__sample_weight": pw}))
            R["lr"][0][va] = lr.predict_proba(X_all[va]); R["lr"][1][:] += lr.predict_proba(X_te) / nf
            msg += f" | LR={f1v(R['lr'][0]):.4f}"

        if "lrn" in models:
            lrn = make_pipeline(MaxAbsScaler(), Normalizer(norm="l2"), LogisticRegression(
                C=CFG["LR_NORM_C"], max_iter=500, class_weight="balanced" if CFG["BALANCE"] else None))
            lrn.fit(X_all[tr], y[tr], **({} if pw is None else {"logisticregression__sample_weight": pw}))
            R["lrn"][0][va] = lrn.predict_proba(X_all[va]); R["lrn"][1][:] += lrn.predict_proba(X_te) / nf
            msg += f" | LRnorm={f1v(R['lrn'][0]):.4f}"

        if "lgb" in models:
            po, pt = 0.0, 0.0
            for sd in seeds:
                g = lgb.LGBMClassifier(
                    objective="multiclass", n_estimators=CFG["LGB_N_EST"], learning_rate=0.08, num_leaves=31,
                    min_child_samples=10, subsample=0.8, subsample_freq=1, colsample_bytree=0.3, reg_lambda=1.0,
                    random_state=sd, n_jobs=-1, verbose=-1, deterministic=True, force_row_wise=True)
                g.fit(Xt[tr], y[tr], sample_weight=sw, eval_set=[(Xt[va], y[va])], callbacks=[lgb.early_stopping(50, verbose=False)])
                po = po + g.predict_proba(Xt[va]) / len(seeds); pt = pt + g.predict_proba(Xt_te) / len(seeds)
            R["lgb"][0][va] = po; R["lgb"][1][:] += pt / nf
            msg += f" | LGBM={f1v(R['lgb'][0]):.4f}"

        if verbose: print(msg)
    return R, imp


### 4-2. 자동 ablation (`AUTO_ABLATION`) — 그룹 분할 OOF로 피처 플래그 선택
후보 7개(`USE_GENE_SPLIT`, `USE_BLOSUM`, `USE_AA_PHYS`, `USE_HS_ALLELE`, `USE_PW_TYPE`, `USE_PW_PAIR`, `USE_NB_STACK`)를 **하나씩** 켜서 그룹 분할 OOF macroF1(XGB 0.5 + 행정규화 LR 0.5)이 기준 대비 `ABL_MIN_GAIN` 이상 오르는지 본다.
이득이 있는 것들을 함께 켜서 한 번 더 검증하고, 합쳐서 오히려 나빠지면 가장 좋은 하나만 유지한다. 그룹 분할을 쓰는 이유는 쌍둥이 중복이 점수를 부풀리지 않게 하기 위해서다.
CV를 (후보 수 + 2)번 돌리므로 오래 걸린다. 끄려면 `CFG["AUTO_ABLATION"]=False`로 두고 플래그를 직접 정한다.


In [17]:
import time
FEATURE_FLAGS = ["USE_GENE_SPLIT", "USE_BLOSUM", "USE_AA_PHYS", "USE_HS_ALLELE", "USE_PW_TYPE", "USE_PW_PAIR", "USE_NB_STACK"]
abl_table = {}

def quick_score():
    Rq, _ = run_cv(ex_tr, ex_te, verbose=False, folds=FOLDS_GROUP, models=("xgb", "lrn"), fast=CFG["ABL_FAST"])
    e = 0.5 * Rq["xgb"][0] + 0.5 * Rq["lrn"][0]
    return macro_f1(y, e.argmax(1), n_class)

if CFG["AUTO_ABLATION"]:
    t0 = time.time()
    _keep_lrn = CFG["USE_LR_NORM"]; CFG["USE_LR_NORM"] = True
    for f_ in FEATURE_FLAGS: CFG[f_] = False
    base = quick_score(); abl_table["기본"] = base
    print(f"기본(전부 OFF) 그룹분할 macroF1 = {base:.4f}   ({time.time() - t0:.0f}s)")
    gains = {}
    for f_ in FEATURE_FLAGS:
        CFG[f_] = True; sc = quick_score(); CFG[f_] = False
        gains[f_] = sc - base; abl_table[f_] = sc
        print(f"  +{f_:<16} macroF1 = {sc:.4f}  ({sc - base:+.4f})   [{time.time() - t0:.0f}s]")
    chosen = [f_ for f_, g_ in gains.items() if g_ >= CFG["ABL_MIN_GAIN"]]
    if len(chosen) > 1:
        for f_ in chosen: CFG[f_] = True
        s_all = quick_score(); abl_table["채택 묶음"] = s_all
        best1 = max(chosen, key=gains.get)
        print(f"  묶음 {chosen} macroF1 = {s_all:.4f} ({s_all - base:+.4f}) | 단독 최고 {best1} ({gains[best1]:+.4f})")
        if s_all - base < gains[best1] - 0.001:
            for f_ in chosen: CFG[f_] = False
            CFG[best1] = True; chosen = [best1]
            print("  -> 합치면 더 나빠서 가장 좋은 하나만 유지")
    elif chosen:
        CFG[chosen[0]] = True
    CFG["USE_LR_NORM"] = _keep_lrn
    print("선택된 플래그:", chosen if chosen else "없음 (기존 피처만 사용)", f"| 총 {time.time() - t0:.0f}s")
else:
    print("자동 ablation 건너뜀. 현재 플래그:", {f_: CFG[f_] for f_ in FEATURE_FLAGS})


기본(전부 OFF) 그룹분할 macroF1 = 0.4933   (333s)
  +USE_GENE_SPLIT   macroF1 = 0.4878  (-0.0054)   [999s]
  +USE_BLOSUM       macroF1 = 0.4941  (+0.0008)   [1327s]
  +USE_AA_PHYS      macroF1 = 0.4891  (-0.0042)   [1685s]
  +USE_HS_ALLELE    macroF1 = 0.4999  (+0.0066)   [2021s]
  +USE_PW_TYPE      macroF1 = 0.4980  (+0.0048)   [2312s]
  +USE_PW_PAIR      macroF1 = 0.4945  (+0.0013)   [2622s]
  +USE_NB_STACK     macroF1 = 0.4870  (-0.0063)   [2946s]
  묶음 ['USE_HS_ALLELE', 'USE_PW_TYPE'] macroF1 = 0.4961 (+0.0028) | 단독 최고 USE_HS_ALLELE (+0.0066)
  -> 합치면 더 나빠서 가장 좋은 하나만 유지
선택된 플래그: ['USE_HS_ALLELE'] | 총 3234s


In [18]:
R, imp = run_cv(ex_tr, ex_te)
print("사용 모델:", list(R.keys()), "| 켜진 피처 플래그:", [f_ for f_ in FEATURE_FLAGS if CFG[f_]])


fold0 | XGB macroF1=0.4520 | LR=0.4280 | LRnorm=0.4102 | LGBM=0.4585
fold1 | XGB macroF1=0.4639 | LR=0.4169 | LRnorm=0.4084 | LGBM=0.4578
fold2 | XGB macroF1=0.4544 | LR=0.4258 | LRnorm=0.3967 | LGBM=0.4539
fold3 | XGB macroF1=0.4706 | LR=0.4170 | LRnorm=0.3961 | LGBM=0.4751
fold4 | XGB macroF1=0.4747 | LR=0.4394 | LRnorm=0.4153 | LGBM=0.4492
사용 모델: ['xgb', 'lr', 'lrn', 'lgb'] | 켜진 피처 플래그: ['USE_HS_ALLELE']


## 5. 평가 · 앙상블 · 오분류 분석

In [19]:
def report(name, p):
    print(f"{name:12s} acc={accuracy_score(y, p.argmax(1)):.4f}  macroF1={macro_f1(y, p.argmax(1), n_class):.4f}")

ms_on = list(R.keys())
OOF = {m: R[m][0] for m in ms_on}; TE = {m: R[m][1] for m in ms_on}
for m in ms_on: report(m.upper(), OOF[m])

def grid(K, n=10):
    out_, cur = [], []
    def rec(i, left):
        if i == K - 1:
            out_.append(tuple(cur + [left / n])); return
        for a in range(left + 1):
            cur.append(a / n); rec(i + 1, left - a); cur.pop()
    rec(0, n); return out_

def blend(P, ms, w): return sum(wi * P[m] for wi, m in zip(w, ms))

def best_w(ms, rows):
    sl = {m: OOF[m][rows] for m in ms}
    return max(grid(len(ms)), key=lambda w: macro_f1(y[rows], blend(sl, ms, w).argmax(1), n_class))

def nested_score(ms):
    """가중치를 '다른 fold'에서 고르고 남은 fold로 평가 -> 가중치 과적합이 빠진 점수."""
    pred = np.zeros(len(y), int)
    for tr_i, va_i in FOLDS:
        w = best_w(ms, tr_i)
        pred[va_i] = blend({m: OOF[m][va_i] for m in ms}, ms, w).argmax(1)
    return macro_f1(y, pred, n_class)

base_ms = [m for m in ("xgb", "lr") if m in ms_on]
cands = {tuple(base_ms): None}
for ms_ in [("xgb", "lrn"), ("xgb", "lr", "lrn"), ("xgb", "lgb", "lr"), ("xgb", "lgb", "lrn"), tuple(ms_on)]:
    if all(m in ms_on for m in ms_) and len(set(ms_)) > 1: cands[tuple(ms_)] = None
for k_ in cands: cands[k_] = nested_score(list(k_))
print("\n조합별 중첩 검증 macroF1 (가중치를 다른 fold에서 고름):")
for k_, v_ in sorted(cands.items(), key=lambda kv: -kv[1]): print(f"  {str(k_):<32}{v_:.4f}")
base_sc = cands[tuple(base_ms)]
best_k = max(cands, key=cands.get)
chosen_ms = list(best_k) if cands[best_k] - base_sc >= CFG["BOOST_MIN_GAIN"] else base_ms
print(f"-> 기준(XGB+LR) 대비 이득이 BOOST_MIN_GAIN({CFG['BOOST_MIN_GAIN']}) 이상인 조합만 채택. 선택: {chosen_ms}")

w_sel = best_w(chosen_ms, np.arange(len(y)))
ens_w = {m: 0.0 for m in ms_on}; ens_w.update(dict(zip(chosen_ms, w_sel)))
ens_t = tuple(ens_w[m] for m in ms_on); best_w_xgb = ens_w.get("xgb", 0.0)
oof_ens = sum(ens_w[m] * OOF[m] for m in ms_on)
te_ens = sum(ens_w[m] * TE[m] for m in ms_on)
print("앙상블 가중치:", {k_: round(v_, 2) for k_, v_ in ens_w.items()})
report("ENSEMBLE", oof_ens)

print("\n" + classification_report(y, oof_ens.argmax(1), target_names=le.classes_, digits=3))
cm = confusion_matrix(y, oof_ens.argmax(1)); np.fill_diagonal(cm, 0)
pairs = sorted(((cm[i, j], le.classes_[i], le.classes_[j]) for i in range(n_class) for j in range(n_class) if cm[i, j] > 0), reverse=True)[:10]
print("가장 많이 헷갈리는 쌍 (정답 -> 예측):")
for c, a, b in pairs: print(f"  {a} -> {b}: {c}")
print("\nXGB 중요도 상위 20:", [k for k, _ in imp.most_common(20)])


XGB          acc=0.4475  macroF1=0.4634
LR           acc=0.4148  macroF1=0.4268
LRN          acc=0.3988  macroF1=0.4068
LGB          acc=0.4448  macroF1=0.4599

조합별 중첩 검증 macroF1 (가중치를 다른 fold에서 고름):
  ('xgb', 'lr', 'lrn', 'lgb')     0.4765
  ('xgb', 'lgb', 'lr')            0.4746
  ('xgb', 'lrn')                  0.4736
  ('xgb', 'lr')                   0.4710
  ('xgb', 'lr', 'lrn')            0.4694
  ('xgb', 'lgb', 'lrn')           0.4682
-> 기준(XGB+LR) 대비 이득이 BOOST_MIN_GAIN(0.003) 이상인 조합만 채택. 선택: ['xgb', 'lr', 'lrn', 'lgb']
앙상블 가중치: {'xgb': 0.0, 'lr': 0.4, 'lrn': 0.1, 'lgb': 0.5}
ENSEMBLE     acc=0.4543  macroF1=0.4791

              precision    recall  f1-score   support

         ACC      0.934     0.792     0.857        72
        BLCA      0.627     0.452     0.525       104
        BRCA      0.710     0.461     0.559       786
        CESC      0.387     0.406     0.396       155
        COAD      0.816     0.735     0.774       223
        DLBC      0.929     0.342     0.500 

In [20]:
# COAD 비율 보정: 검토 결과 미적용(k=1.0) → test 정답 분포 정보를 쓰지 않기 위해 삭제 (11/12)
te_final = te_ens.copy()


In [21]:
# ---- 현업 기준 점수: 같은 프로필을 같은 fold에 묶은 그룹 분할 OOF (주 CV가 무작위 분할일 때만) ----
honest = None
if CFG["HONEST_REPORT"] and not CFG["GROUP_CV"]:
    Rg, _ = run_cv(ex_tr, ex_te, verbose=False, folds=FOLDS_GROUP)
    _ge = sum(ens_w.get(m, 0.0) * Rg[m][0] for m in Rg)       # 주 CV에서 고른 가중치 그대로
    honest = dict(macroF1=float(macro_f1(y, _ge.argmax(1), n_class)), acc=float(accuracy_score(y, _ge.argmax(1))))
    print(f"[현업 기준] 그룹 분할 OOF  macroF1={honest['macroF1']:.4f}  acc={honest['acc']:.4f}   (같은 환자/프로필이 학습과 검증에 갈라지지 않음)")
    print(f"[대회 조건] 무작위 분할 OOF macroF1={macro_f1(y, oof_ens.argmax(1), n_class):.4f}   <- 두 점수의 차이가 중복 구조의 영향")


[현업 기준] 그룹 분할 OOF  macroF1=0.5052  acc=0.5022   (같은 환자/프로필이 학습과 검증에 갈라지지 않음)
[대회 조건] 무작위 분할 OOF macroF1=0.4791   <- 두 점수의 차이가 중복 구조의 영향


## 5-2. 학습형 중복 규칙 (`USE_DUP_RULE`)

- 같은 환자가 개별 코호트(`KIRC`)와 통합 코호트(`KIPAN`)에 모두 등록되면 변이 프로필이 같은 행이 둘 생긴다. 한 행만 train에 있으면 모델은 그 라벨을 그대로 찍어 틀린다 (fixed4 출력: 해당 행 정답률 0.096).
- 이전 규칙은 암종 이름을 코드에 박아 둔 `TWIN_SWAP`이었다. 여기서는 **표를 train에서 추정**한다: "같은 프로필을 가진 *다른* train 행들의 라벨 묶음 -> 이 행의 라벨 분포"를 leave-one-out으로 센다. 한 칸의 표본 수가 `DUP_MIN_SUPPORT` 이상이고 최다 라벨 비율이 `DUP_MIN_PROB` 이상일 때만 쓴다.
- 장점: 암종 이름이 코드에 없다(어떤 쌍에도 적용), 근거가 부족하면 자동으로 꺼진다, 표를 CSV로 감사할 수 있다.
- **한계(솔직히)**: 효과의 출처는 이전과 같은 "이중 등록 구조"다. 새 환자에게는 이런 짝이 없으므로 **현업 성능이 아니다.** 그래서 제출 파일을 규칙 없는 `_core`와 규칙 적용 `_dup`으로 나눠 만든다. `_dup`을 내도 되는지는 대회 규정 확인이 필요하다.
- 검증: 주 CV가 무작위 분할일 때만 시뮬이 가능하다 (그룹 분할이면 짝이 검증에 남지 않아 발동 0). 사전은 fold의 train 행으로만 만들고 검증 행은 한 행씩 독립으로 찾아본다. test끼리는 비교하지 않는다.


In [22]:
def build_dup_table(h, lab):
    """같은 프로필 행들 중 '나를 뺀 나머지의 라벨 묶음' -> 내 라벨 분포 (leave-one-out)."""
    cnt = {}
    for labs in pd.Series(np.asarray(lab)).groupby(np.asarray(h)).agg(list):
        if len(labs) < 2 or len(labs) - 1 > CFG["DUP_MAX_CTX"]:
            continue
        for k, yk in enumerate(labs):
            cnt.setdefault(tuple(sorted(labs[:k] + labs[k + 1:])), Counter())[yk] += 1
    tab = {}
    for ctx, c in cnt.items():
        n = sum(c.values()); lab_, k_ = c.most_common(1)[0]
        if n >= CFG["DUP_MIN_SUPPORT"] and k_ / n >= CFG["DUP_MIN_PROB"]:
            tab[ctx] = (lab_, k_ / n, n)
    return tab

def dup_override(tab, tr_hash, tr_labels, q_hash):
    """질의 행마다 바꿀 암종 번호(없으면 -1). train 프로필->라벨 목록 사전을 만들고 한 행씩 찾아본다."""
    labs = pd.Series(np.asarray(tr_labels)).groupby(np.asarray(tr_hash)).agg(list).to_dict()
    cls = list(le.classes_); out = np.full(len(q_hash), -1)
    for i, h in enumerate(q_hash):
        L = labs.get(h)
        if L and len(L) <= CFG["DUP_MAX_CTX"]:
            d = tab.get(tuple(sorted(L)))
            if d is not None and d[0] in cls:
                out[i] = cls.index(d[0])
    return out

def apply_override(P, ov):
    """ov >= 0 인 행은 그 암종 확률을 1로(나머지 0), 나머지 행은 그대로."""
    Q = P.copy(); m = ov >= 0
    Q[m] = 0.0; Q[m, ov[m]] = 1.0
    return Q

# ---- 동작 확인 (작은 예제: 30쌍 KIRC/KIPAN, 25쌍 LGG/GBMLGG, 잡음 군집) ----
_keep = {k: CFG[k] for k in ("DUP_MIN_SUPPORT", "DUP_MIN_PROB", "DUP_MAX_CTX")}
CFG.update(DUP_MIN_SUPPORT=20, DUP_MIN_PROB=0.9, DUP_MAX_CTX=3)
_h, _l = [], []
for k in range(30): _h += [f"k{k}"] * 2; _l += ["KIRC", "KIPAN"]
for k in range(25): _h += [f"g{k}"] * 2; _l += ["LGG", "GBMLGG"]
for k in range(5):  _h += [f"x{k}"] * 2; _l += ["BRCA", "OV"]          # 표본 부족 -> 쓰이지 않아야 함
_tab = build_dup_table(_h, _l)
assert _tab[("KIRC",)][0] == "KIPAN" and _tab[("KIPAN",)][0] == "KIRC" and _tab[("LGG",)][0] == "GBMLGG"
assert ("BRCA",) not in _tab and ("OV",) not in _tab                   # 표본 수 미달 칸은 제외
_le_bak, le = le, type("L", (), {"classes_": np.array(["BRCA", "GBMLGG", "KIPAN", "KIRC", "LGG", "OV"])})()
_trh, _trl = ["a", "b", "b", "c"], ["KIRC", "KIRC", "KIPAN", "BRCA"]    # a: 짝 하나만 train에 있음 / b: 짝이 이미 둘 다 train / c: 표에 없음
assert dup_override(_tab, _trh, _trl, ["a", "b", "c", "z"]).tolist() == [2, -1, -1, -1]
le = _le_bak; CFG.update(_keep)
assert np.allclose(apply_override(np.full((3, 5), 0.2), np.array([-1, 2, 0])).sum(1), 1)

# ---- 시뮬(진단): 주 CV와 같은 FOLDS, fold의 학습 부분만으로 표를 만들어 검증 부분에 적용 ----
sim = oof_ens.copy(); hit_oof = np.zeros(len(y), bool)
for tr_i, va_i in FOLDS:
    tab_f = build_dup_table(H_tr[tr_i], lab_tr[tr_i])
    ov = dup_override(tab_f, H_tr[tr_i], lab_tr[tr_i], H_tr[va_i])
    sim[va_i] = apply_override(oof_ens[va_i], ov); hit_oof[va_i[ov >= 0]] = True
n_hit = int(hit_oof.sum())
f0, f1 = (f1_score(y, p.argmax(1), average="macro") for p in (oof_ens, sim))
print(f"중복 규칙 시뮬: OOF macroF1 {f0:.4f} -> {f1:.4f} ({f1 - f0:+.4f}) | 발동 {n_hit}행 ({n_hit/len(y):.1%})")
if n_hit:
    print(f"  발동 행 정답률 {(sim[hit_oof].argmax(1) == y[hit_oof]).mean():.3f} (규칙 전 {(oof_ens[hit_oof].argmax(1) == y[hit_oof]).mean():.3f})")
elif CFG["GROUP_CV"]:
    print("  (그룹 분할이라 같은 프로필이 검증 fold에 짝으로 남지 않음 -> 시뮬 불가. 정상)")

# ---- 전체 train으로 만든 표(감사용 출력)와 test 쪽 규모 ----
dup_tab = build_dup_table(H_tr, lab_tr)
print("학습된 중복 표 (같은 프로필의 다른 train 라벨 묶음 -> 예측 라벨, 비율, 표본 수):")
for ctx, (lb, p, n) in sorted(dup_tab.items(), key=lambda kv: -kv[1][2]):
    print(f"  {ctx} -> {lb}  p={p:.3f}  n={n}")
ov_te = dup_override(dup_tab, H_tr, lab_tr, H_te)
print(f"test에서 발동 가능한 행: {(ov_te >= 0).sum()}행 ({(ov_te >= 0).mean():.1%}) ->",
      pd.Series(le.classes_[ov_te[ov_te >= 0]]).value_counts().to_dict())
dup_applied = bool(CFG["USE_DUP_RULE"] and n_hit >= CFG["DUP_MIN_SUPPORT"] and (f1 - f0) >= CFG["BOOST_MIN_GAIN"])
print("중복 규칙", "적용 (제출 파일 _dup)" if dup_applied else "미적용 (OFF이거나 시뮬 이득/표본이 기준 미만)")


중복 규칙 시뮬: OOF macroF1 0.4791 -> 0.5451 (+0.0660) | 발동 716행 (11.5%)
  발동 행 정답률 0.997 (규칙 전 0.066)
학습된 중복 표 (같은 프로필의 다른 train 라벨 묶음 -> 예측 라벨, 비율, 표본 수):
  ('KIPAN',) -> KIRC  p=0.993  n=272
  ('KIRC',) -> KIPAN  p=1.000  n=270
  ('GBMLGG',) -> LGG  p=1.000  n=173
  ('LGG',) -> GBMLGG  p=1.000  n=173
test에서 발동 가능한 행: 220행 (8.6%) -> {'KIRC': 62, 'KIPAN': 58, 'GBMLGG': 52, 'LGG': 48}
중복 규칙 적용 (제출 파일 _dup)


## 5-3. macroF1 클래스별 확률 배율 보정 (`USE_CLASS_SCALE`)

macroF1은 클래스별 F1의 평균이라, 판정 기준을 클래스마다 조금씩 옮기면(확률에 배율을 곱한 뒤 argmax) 점수가 오를 수 있다. 의료 분류에서도 쓰는 표준적인 결정 임계값 조정이다.
- 배율은 **OOF 확률과 train 라벨만**으로 맞춘다 (test 분포 정보 사용 안 함. 이전에 뺀 EM 사전확률 보정과 다름).
- 과적합 방지: fold마다 "나머지 fold의 OOF로 배율을 맞추고 남은 fold로 평가"하는 **중첩 검증**을 하고, 그 이득이 `BOOST_MIN_GAIN` 이상일 때만 적용한다. 보고하는 이득은 이 보수적인 추정치다.
- 중복 규칙이 정답을 바로잡는 행은 배율을 맞출 때 제외한다 (규칙이 이미 해결하는 오류에 배율이 맞춰지는 것을 막기 위해).


In [23]:
def macro_f1(yt, pred, K):
    tp = np.bincount(yt[pred == yt], minlength=K).astype(float)
    pc, tc = np.bincount(pred, minlength=K), np.bincount(yt, minlength=K)
    d = pc + tc
    return float(np.mean(np.where(d > 0, 2 * tp / np.maximum(d, 1), 0.0)[d > 0]))

_r = np.random.default_rng(0); _yt = _r.integers(0, 6, 500); _pr = _r.integers(0, 6, 500)   # sklearn과 같은 값인지 확인
assert abs(macro_f1(_yt, _pr, 6) - f1_score(_yt, _pr, average="macro")) < 1e-9

SCALE_GRID = np.exp(np.linspace(-1.0, 1.0, 21))

def tune_scale(P, yy, iters):
    """클래스 하나씩 배율을 바꿔 보며 macroF1이 오르면 채택 (좌표 탐색). 기준은 배율 1.0."""
    K = P.shape[1]; s = np.ones(K); best = macro_f1(yy, P.argmax(1), K)
    for _ in range(iters):
        for c in range(K):
            for g in SCALE_GRID:
                if g == s[c]: continue
                t = s.copy(); t[c] = g
                v = macro_f1(yy, (P * t).argmax(1), K)
                if v > best + 1e-9: best, s = v, t
    return s, best

# ---- 중첩 검증: 다른 fold로 배율을 맞추고, 남은 fold의 (중복 규칙이 안 건드리는) 행으로 평가 ----
mask = ~hit_oof if dup_applied else np.ones(len(y), bool)
pred_raw, pred_scaled, y_eval = [], [], []
for tr_i, va_i in FOLDS:
    fit_rows = tr_i[mask[tr_i]]; ev_rows = va_i[mask[va_i]]
    s_f, _ = tune_scale(oof_ens[fit_rows], y[fit_rows], CFG["SCALE_ITERS"])
    pred_raw.append(oof_ens[ev_rows].argmax(1)); pred_scaled.append((oof_ens[ev_rows] * s_f).argmax(1)); y_eval.append(y[ev_rows])
pred_raw, pred_scaled, y_eval = map(np.concatenate, (pred_raw, pred_scaled, y_eval))
g0, g1 = macro_f1(y_eval, pred_raw, n_class), macro_f1(y_eval, pred_scaled, n_class)
scale_applied = bool(CFG["USE_CLASS_SCALE"] and (g1 - g0) >= CFG["BOOST_MIN_GAIN"])
print(f"클래스 배율 중첩 검증: macroF1 {g0:.4f} -> {g1:.4f} ({g1 - g0:+.4f}) | 적용: {scale_applied}")

scale = np.ones(n_class)
if scale_applied:
    scale, _s_in = tune_scale(oof_ens[mask], y[mask], CFG["SCALE_ITERS"])      # 최종 배율은 전체 OOF로
    print("배율(1.0 아님):", {le.classes_[c]: round(float(v), 2) for c, v in enumerate(scale) if abs(v - 1) > 1e-6})
oof_core, te_core = oof_ens * scale, te_ens * scale
te_dup = apply_override(te_core, ov_te) if dup_applied else te_core
oof_dup = apply_override(oof_core, np.where(hit_oof, sim.argmax(1), -1)) if dup_applied else oof_core
print(f"OOF macroF1 (참고): 앙상블 {f1_score(y, oof_ens.argmax(1), average='macro'):.4f} | 배율 적용 {f1_score(y, oof_core.argmax(1), average='macro'):.4f} (배율은 같은 OOF로 맞춘 값이라 낙관적, 위 중첩 검증값이 보수적 추정)"
      + (f" | +중복 규칙 {f1_score(y, oof_dup.argmax(1), average='macro'):.4f}" if dup_applied else ""))


클래스 배율 중첩 검증: macroF1 0.5032 -> 0.4910 (-0.0121) | 적용: False
OOF macroF1 (참고): 앙상블 0.4791 | 배율 적용 0.4791 (배율은 같은 OOF로 맞춘 값이라 낙관적, 위 중첩 검증값이 보수적 추정) | +중복 규칙 0.5451


## 5-4. 라벨 분포 이동 보정 (BBSE, `USE_LABEL_SHIFT`) — 별도 `_shift` 제출본

**배경**: v12 출력에서 test 예측이 COAD 548행, STES 377행으로 쏠려 있다 (train 비율로는 각각 약 92, 156행 수준). train과 test의 클래스 비율이 다르다는 신호다.
이전 EM 사전확률 보정이 macroF1을 0.46 → 0.21로 망가뜨린 이유는, 이 모델이 **balanced 가중치로 학습돼 확률이 train 비율을 반영하지 않기 때문**일 가능성이 크다 (EM은 확률이 train 사전확률 기준이라고 가정한다).

여기서는 EM 대신 **BBSE**(Lipton et al. 2018)를 쓴다: OOF 혼동행렬(`P(예측, 정답)`)과 test의 **예측 분포**만으로 클래스 비율비 `w = q(y)/p(y)`를 추정한다. 확률 보정에 의존하지 않는다.
- 후보 보정 4종(`w^0.5`, `w`, `q^0.5`, `q`; `q = w·p`)을 **OOF 재표집 시뮬레이션**으로 평가한다: test와 같은 클래스 비율이 되도록 OOF를 중요도 재표집하고, 가중치는 재표집에 쓰지 않은 절반의 OOF로 추정한다.
- 시뮬 평균 이득이 `BOOST_MIN_GAIN` 이상이고 반복의 70% 이상에서 이득일 때만 적용한다. 비율 이동이 없는 경우(null)에도 해가 없는지 같이 본다.
- **주의**: test의 *피처*(라벨 아님)로 만든 예측 분포를 쓴다. 이전에 "test 분포 정보가 필요하다"며 뺀 것과 같은 성격이라, 기존 `_core`/`_dup`은 그대로 두고 `_shift` 파일을 **따로** 만든다. 낼지는 대회 규정 확인 후 선택.


In [24]:
from scipy.optimize import nnls

def bbse_weights(pred_src, y_src, pred_tgt, K, clip=(0.25, 4.0)):
    """C[i,j] = P(pred=i, y=j) (source), mu_i = P(pred=i) (target). C w = mu, w>=0 의 해 -> w_j = q(j)/p(j)."""
    C = np.zeros((K, K))
    np.add.at(C, (pred_src, y_src), 1.0); C /= len(y_src)
    mu = np.bincount(pred_tgt, minlength=K) / len(pred_tgt)
    w, _ = nnls(C, mu)
    return np.clip(w, *clip)

te_shift_core, te_shift_dup, shift_applied, shift_best = None, None, False, None
if CFG["USE_LABEL_SHIFT"]:
    K = n_class
    p_tr = np.bincount(y, minlength=K) / len(y)
    pred_oof, pred_te = oof_core.argmax(1), te_core.argmax(1)
    w_hat = bbse_weights(pred_oof, y, pred_te, K)
    q_hat = w_hat * p_tr; q_hat = q_hat / q_hat.sum()
    te_dist = np.bincount(pred_te, minlength=K) / len(pred_te)
    print(f"{'클래스':<8}{'train비율':>10}{'test예측비율':>13}{'추정 test비율':>14}{'w':>7}")
    for c in np.argsort(-np.abs(q_hat - p_tr))[:8]:
        print(f"{le.classes_[c]:<8}{p_tr[c]:>10.3f}{te_dist[c]:>13.3f}{q_hat[c]:>14.3f}{w_hat[c]:>7.2f}")

    def make_mults(w, q):
        return {"w^0.5": w ** 0.5, "w^1": w, "q^0.5": q ** 0.5, "q^1": q}

    def simulate(target_q, reps=30, seed=0):
        rg = np.random.default_rng(seed); n_s = min(len(te_core), len(y) // 2)
        res = {k_: [] for k_ in make_mults(w_hat, q_hat)}
        for _ in range(reps):
            perm = rg.permutation(len(y)); A, B = perm[:len(y) // 2], perm[len(y) // 2:]
            pb = target_q[y[B]] / p_tr[y[B]]; pb = pb / pb.sum()
            S_ = rg.choice(B, size=n_s, replace=True, p=pb)               # 목표 비율을 가진 가짜 test
            wA = bbse_weights(pred_oof[A], y[A], oof_core[S_].argmax(1), K)  # 추정은 가짜 test의 '예측'만 사용
            qA = wA * p_tr; qA = qA / qA.sum()
            base_f = macro_f1(y[S_], oof_core[S_].argmax(1), K)
            for k_, mu_ in make_mults(wA, qA).items():
                res[k_].append(macro_f1(y[S_], (oof_core[S_] * mu_).argmax(1), K) - base_f)
        return {k_: (float(np.mean(v_)), float(np.mean(np.array(v_) > 0))) for k_, v_ in res.items()}

    sim_shift, sim_null = simulate(q_hat), simulate(p_tr, seed=1)
    print("\n시뮬 (이동 있음: test 추정 비율로 재표집)  보정별 [평균 macroF1 이득, 이득 반복 비율]")
    for k_, (g_, wr_) in sim_shift.items(): print(f"  {k_:<7} {g_:+.4f}  승률 {wr_:.0%}   | null(이동 없음) {sim_null[k_][0]:+.4f}")
    shift_best = max(sim_shift, key=lambda k_: sim_shift[k_][0])
    g_, wr_ = sim_shift[shift_best]
    shift_applied = bool(g_ >= CFG["BOOST_MIN_GAIN"] and wr_ >= 0.7 and sim_null[shift_best][0] > -CFG["BOOST_MIN_GAIN"])
    print(f"-> 최고 후보 {shift_best}: 이득 {g_:+.4f}, 승률 {wr_:.0%} | 적용: {shift_applied}")
    if shift_applied:
        mu_final = make_mults(w_hat, q_hat)[shift_best]
        te_shift_core = te_core * mu_final
        te_shift_dup = apply_override(te_shift_core, ov_te) if dup_applied else None
        print("보정 후 test 예측 상위:", pd.Series(le.classes_[te_shift_core.argmax(1)]).value_counts().head(6).to_dict())
else:
    print("라벨 이동 보정 건너뜀")


클래스        train비율     test예측비율     추정 test비율      w
STES         0.061        0.162         0.240   3.70
COAD         0.036        0.209         0.153   4.00
GBMLGG       0.074        0.040         0.020   0.25
KIPAN        0.083        0.086         0.030   0.34
BRCA         0.127        0.059         0.085   0.63
KIRC         0.054        0.035         0.085   1.49
HNSC         0.036        0.021         0.010   0.25
LGG          0.037        0.027         0.062   1.59

시뮬 (이동 있음: test 추정 비율로 재표집)  보정별 [평균 macroF1 이득, 이득 반복 비율]
  w^0.5   +0.0119  승률 77%   | null(이동 없음) -0.0194
  w^1     -0.0364  승률 0%   | null(이동 없음) -0.0436
  q^0.5   -0.0151  승률 13%   | null(이동 없음) -0.0416
  q^1     -0.1094  승률 0%   | null(이동 없음) -0.0994
-> 최고 후보 w^0.5: 이득 +0.0119, 승률 77% | 적용: False


## 6. 제출 파일 생성

In [25]:
import json, sys, platform, datetime

tag = "norm" if CFG["NORMALIZE_TOKENS"] else "nonorm"

def save_submission(P, suffix):
    pred = le.inverse_transform(P.argmax(1))
    out = sub.copy(); out["SUBCLASS"] = pred
    assert len(out) == len(test) and set(out["SUBCLASS"]) <= set(le.classes_)
    path = f"{D}/submission_v13_{tag}_{suffix}.csv"
    out.to_csv(path, index=False, encoding="UTF-8-sig")
    np.save(f"{D}/te_final_v13_{tag}_{suffix}.npy", P)
    print(f"저장: {path}")
    return out

out_core = save_submission(te_core, "core")                 # 현업형: 모델 + (검증된) 클래스 배율. 쌍둥이/중복 덮어쓰기 없음
out_dup = save_submission(te_dup, "dup") if dup_applied else None   # 대회 조건형: + 학습형 중복 규칙
out_shift_core = save_submission(te_shift_core, "core_shift") if shift_applied else None   # + 라벨 분포 이동 보정 (규정 확인 후 선택)
out_shift_dup = save_submission(te_shift_dup, "dup_shift") if (shift_applied and te_shift_dup is not None) else None
np.save(f"{D}/oof_ens_v13_{tag}.npy", oof_ens)
pd.DataFrame([(str(c), v[0], v[1], v[2]) for c, v in dup_tab.items()], columns=["context", "label", "prob", "n"]).to_csv(
    f"{D}/dup_table_v13_{tag}.csv", index=False, encoding="UTF-8-sig")

def _md5(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()

manifest = dict(
    created=datetime.datetime.now().isoformat(timespec="seconds"),
    cfg={k: (v if isinstance(v, (int, float, str, bool)) else str(v)) for k, v in CFG.items()},
    versions=dict(python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__,
                  xgboost=xgb.__version__, lightgbm=(lgb.__version__ if lgb else None), platform=platform.platform(),
                  sklearn=__import__("sklearn").__version__),
    data=dict(train_md5=_md5(f"{D}/train.csv"), test_md5=_md5(f"{D}/test.csv"), n_train=len(train), n_test=len(test)),
    scores=dict(oof_ens_macroF1_main_cv=float(f1_score(y, oof_ens.argmax(1), average="macro")),
                honest_group_cv=honest, class_scale_nested=dict(before=g0, after=g1, applied=scale_applied),
                dup_rule_sim=dict(before=float(f0), after=float(f1), n_hit=n_hit, applied=dup_applied)),
    ensemble_weights=ens_w,
    features_on={f_: bool(CFG[f_]) for f_ in FEATURE_FLAGS}, ablation_table=abl_table,
    label_shift=dict(applied=bool(shift_applied), best=shift_best),
    outputs=dict(core=out_core["SUBCLASS"].value_counts().to_dict(),
                 dup=(out_dup["SUBCLASS"].value_counts().to_dict() if out_dup is not None else None)),
)
with open(f"{D}/run_manifest_v13_{tag}.json", "w", encoding="utf-8") as fp:
    json.dump(manifest, fp, ensure_ascii=False, indent=2, default=str)
print("저장: run_manifest_v13_%s.json (재현에 필요한 설정·버전·데이터 해시·점수)" % tag)
print(out_core["SUBCLASS"].value_counts().head(10))


저장: ../../data/submission_v13_norm_core.csv
저장: ../../data/submission_v13_norm_dup.csv
저장: run_manifest_v13_norm.json (재현에 필요한 설정·버전·데이터 해시·점수)
SUBCLASS
COAD      533
STES      412
KIPAN     219
BRCA      150
OV        119
PRAD      109
GBMLGG    102
KIRC       89
SKCM       88
UCEC       77
Name: count, dtype: int64
